In [3]:
pip install  -U bitsandbytes pdfplumber tqdm langchain langchain-openai neo4j networkx graspologic scikit-learn sentence-transformers llama-index rouge-score PyPDF2 openai numpy==1.26.4 scipy==1.12.0 transformers langchain-huggingface torch sentence-transformers


Defaulting to user installation because normal site-packages is not writeable
  Using cached langchain_openai-1.0.1-py3-none-any.whl.metadata (1.8 kB)
  Using cached neo4j-6.0.2-py3-none-any.whl.metadata (5.2 kB)
  Using cached graspologic-3.4.4-py3-none-any.whl.metadata (5.8 kB)
  Using cached scikit_learn-1.7.2-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (11 kB)
  Using cached openai-2.6.0-py3-none-any.whl.metadata (29 kB)
  Using cached transformers-4.57.1-py3-none-any.whl.metadata (43 kB)
  Using cached langchain_huggingface-1.0.0-py3-none-any.whl.metadata (2.1 kB)
  Using cached torch-2.6.0-cp311-cp311-manylinux1_x86_64.whl.metadata (28 kB)
  Using cached langchain_core-1.0.0-py3-none-any.whl.metadata (3.4 kB)
  Using cached hyppo-0.5.2-py3-none-any.whl.metadata (2.1 kB)
  Using cached tokenizers-0.22.1-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.8 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.

In [4]:
!pip install transformers langchain-huggingface torch sentence-transformers bitsandbytes accelerate pdfplumber neo4j networkx graspologic scikit-learn tqdm

Defaulting to user installation because normal site-packages is not writeable


In [5]:
# 1. Remove broken installs
!pip uninstall -y torch torchvision torchaudio transformers sentence-transformers bitsandbytes

# 2. Install a clean PyTorch + TorchVision stack (CUDA 12.1 build for Colab GPUs)
!pip install torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu121

# 3. Install Hugging Face + Sentence Transformers + bitsandbytes
!pip install -U transformers==4.44.2 sentence-transformers accelerate bitsandbytes safetensors

import torch, torchvision, transformers, sentence_transformers
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("transformers:", transformers.__version__)
print("sentence-transformers:", sentence_transformers.__version__)


Found existing installation: torch 2.4.0+cu121
Uninstalling torch-2.4.0+cu121:
  Successfully uninstalled torch-2.4.0+cu121
Found existing installation: torchvision 0.19.0+cu121
Uninstalling torchvision-0.19.0+cu121:
  Successfully uninstalled torchvision-0.19.0+cu121
Found existing installation: torchaudio 2.4.0+cu121
Uninstalling torchaudio-2.4.0+cu121:
  Successfully uninstalled torchaudio-2.4.0+cu121
Found existing installation: transformers 4.44.2
Uninstalling transformers-4.44.2:
  Successfully uninstalled transformers-4.44.2
Found existing installation: sentence-transformers 5.1.2
Uninstalling sentence-transformers-5.1.2:
  Successfully uninstalled sentence-transformers-5.1.2
Found existing installation: bitsandbytes 0.42.0
Uninstalling bitsandbytes-0.42.0:
  Successfully uninstalled bitsandbytes-0.42.0
Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download.pytorch.o

2025-10-24 12:43:27.232028: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-24 12:43:27.301632: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-10-24 12:43:27.301666: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-10-24 12:43:27.301704: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-10-24 12:43:27.318322: I tensorflow/core/platform/cpu_feature_g

torch: 2.4.0+cu121
torchvision: 0.19.0+cu121
transformers: 4.44.2
sentence-transformers: 5.1.2


In [2]:

!pip install future
!pip install langchain-classic


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip


In [ ]:
import os
import pdfplumber
from tqdm import tqdm



import json
import re
from typing import List, Tuple, Dict, Set, Optional, Any
from neo4j import GraphDatabase
import networkx as nx
from graspologic.partition import hierarchical_leiden
from collections import defaultdict
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import heapq
import logging
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_classic.chains import LLMChain

# Set up logging
logging.basicConfig(level=logging.INFO)

class PDFProcessor:
    """Handles PDF processing and text extraction"""

    def __init__(self, pdf_dir: str = './test/'):
        self.pdf_dir = pdf_dir

    def extract_texts(self) -> List[str]:
        """Extract text from all PDFs in the directory"""
        texts = []
        pdf_files = [f for f in os.listdir(self.pdf_dir) if f.endswith('.pdf')]

        for filename in tqdm(pdf_files, desc="Processing PDF files"):
            pdf_path = os.path.join(self.pdf_dir, filename)
            try:
                with pdfplumber.open(pdf_path) as pdf:
                    for page in tqdm(pdf.pages, desc=f"Processing {filename}", leave=False):
                        text = page.extract_text()
                        if text and text.strip():
                            texts.append(text)
            except Exception as e:
                logging.error(f"Error processing {filename}: {e}")

        return texts

    def create_chunks(self, texts: List[str], chunk_size: int = 1024, chunk_overlap: int = 20) -> List[str]:
        """Split texts into chunks and return as strings"""
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap
        )
        documents = splitter.create_documents(texts)
        return [doc.page_content for doc in documents]

In [ ]:
from typing import List, Tuple, Dict
import json
import re
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx
from sklearn.neighbors import NearestNeighbors
import heapq
import logging

class TripletExtractor:
    """Handles knowledge graph triplet extraction"""

    def __init__(self, llm_pipeline):
        self.llm_pipeline = llm_pipeline  # Expecting a transformers pipeline object

    def parse_response(self, response_str: str) -> Tuple[List[tuple], List[tuple]]:
        """Parse LLM response to extract entities and relationships"""
        entities, relationships = [], []

        try:
            # Clean the response by removing any special tokens or extra whitespace
            response_clean = re.sub(r'<\|.*?\|>', '', response_str).strip()

            # Extract JSON using a robust pattern
            json_pattern = r'\{.*\}'
            match = re.search(json_pattern, response_clean, re.DOTALL)
            if not match:
                logging.warning(f"No valid JSON found in response: {response_clean[:100]}...")
                return entities, relationships

            json_str = match.group(0)
            data = json.loads(json_str)

            # Validate and extract entities
            for entity in data.get("entities", []):
                if all(key in entity for key in ["entity_name", "entity_type", "entity_description"]):
                    entities.append((
                        entity["entity_name"],
                        entity["entity_type"],
                        entity["entity_description"],
                        entity.get("entity_attributes", {})
                    ))

            # Validate and extract relationships
            for relation in data.get("relationships", []):
                if all(key in relation for key in ["source_entity", "target_entity", "relation", "relationship_description"]):
                    relationships.append((
                        relation["source_entity"],
                        relation["target_entity"],
                        relation["relation"],
                        relation["relationship_description"],
                        relation.get("relationship_attributes", {})
                    ))

            logging.info(f"Successfully parsed {len(entities)} entities and {len(relationships)} relationships")
            return entities, relationships

        except json.JSONDecodeError as e:
            logging.error(f"JSON parsing error: {e}, Response: {response_clean[:100]}...")
            return entities, relationships
        except Exception as e:
            logging.error(f"Unexpected error in parse_response: {e}, Response: {response_clean[:100]}...")
            return entities, relationships

    def extract_triplets(self, text: str, max_paths_per_chunk: int = 2) -> Tuple[List[tuple], List[tuple]]:
        """Extract triplets from text using raw transformers pipeline"""
        try:
            # Construct prompt with Llama 3.1 chat template
            prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id>
            You are an expert in extracting entity-relation triplets from text related to diabetes. Extract up to {max_paths_per_chunk} triplets from the provided text and return a valid JSON object in this format:

            {{
              "entities": [
                {{
                  "entity_name": "string",
                  "entity_type": "string",
                  "entity_description": "string",
                  "entity_attributes": {{}}
                }}
              ],
              "relationships": [
                {{
                  "source_entity": "string",
                  "target_entity": "string",
                  "relation": "string",
                  "relationship_description": "string",
                  "relationship_attributes": {{}}
                }}
              ]
            }}

            Instructions:
            1. Identify entities with:
              - entity_name: Capitalized name (e.g., Insulin, Diabetes)
              - entity_type: Type (e.g., Drug, Disease, Protein)
              - entity_description: Brief description
              - entity_attributes: Key-value pairs (e.g., {{"domain": "Diabetes"}})
            2. Identify relationships with:
              - source_entity: Source entity name
              - target_entity: Target entity name
              - relation: Relationship type (e.g., TREATS, CAUSES)
              - relationship_description: Brief explanation
              - relationship_attributes: Key-value pairs (e.g., {{"context": "Medical"}})
            3. Return only the JSON object. No additional text or tokens.

            Example:
            {{
              "entities": [
                {{
                  "entity_name": "Insulin",
                  "entity_type": "Drug",
                  "entity_description": "A hormone used to manage diabetes",
                  "entity_attributes": {{"domain": "Diabetes"}}
                }},
                {{
                  "entity_name": "Diabetes",
                  "entity_type": "Disease",
                  "entity_description": "A chronic condition affecting blood sugar",
                  "entity_attributes": {{"domain": "Medical"}}
                }}
              ],
              "relationships": [
                {{
                  "source_entity": "Insulin",
                  "target_entity": "Diabetes",
                  "relation": "TREATS",
                  "relationship_description": "Insulin is used to control blood sugar in diabetes",
                  "relationship_attributes": {{"context": "Medical Treatment"}}
                }}
              ]
            }}

            Input Text:
            {text[:4000]}
            <|end_header_id><|start_header_id|>user<|end_header_id>
            Return the extracted triplets in the specified JSON format.
            <|end_header_id>"""

            # Generate response using the raw pipeline
            response = self.llm_pipeline(prompt, max_new_tokens=512, temperature=0.1, do_sample=True, top_p=0.95, repetition_penalty=1.2)
            response_text = response[0]["generated_text"] if isinstance(response, list) else response
            # Extract the output after the prompt
            response_text = response_text[len(prompt):].strip() if response_text.startswith(prompt) else response_text
            logging.debug(f"Raw LLM response: {response_text[:100]}...")
            return self.parse_response(response_text)
        except Exception as e:
            logging.error(f"Error extracting triplets: {str(e)}, Text: {text[:100]}...")
            return [], []
class ArchRAGCHNSW:
    """C-HNSW Index Implementation for ArchRAG"""

    def __init__(self, embedding_dim: int = 384, M: int = 16, ef_construction: int = 200):
        self.embedding_dim = embedding_dim
        self.M = M
        self.ef_construction = ef_construction
        self.layers = []
        self.node_embeddings = {}
        self.inter_layer_links = {}
        self.node_data = {}
        self.nn_index = None

    def add_node(self, node_id: str, embedding: np.ndarray, layer: int, node_data: dict = None):
        """Add a node to the specified layer"""
        while len(self.layers) <= layer:
            self.layers.append(nx.Graph())

        self.layers[layer].add_node(node_id)
        self.node_embeddings[node_id] = embedding
        if node_data:
            self.node_data[node_id] = node_data

    def distance(self, a: np.ndarray, b: np.ndarray) -> float:
        """Compute cosine distance between two vectors"""
        return 1 - cosine_similarity([a], [b])[0][0]

    def build_intra_layer_links(self):
        """Build links within each layer using approximate nearest neighbors"""
        for layer_idx, layer_graph in enumerate(self.layers):
            nodes = list(layer_graph.nodes())
            if len(nodes) <= 1:
                continue

            embeddings = np.array([self.node_embeddings[n] for n in nodes])
            nn = NearestNeighbors(n_neighbors=min(self.M, len(nodes)), metric="cosine")
            nn.fit(embeddings)

            distances, indices = nn.kneighbors(embeddings)
            for i, node_id in enumerate(nodes):
                for j, dist in zip(indices[i], distances[i]):
                    if j != i:
                        neighbor_id = nodes[j]
                        layer_graph.add_edge(node_id, neighbor_id, weight=dist, similarity=1-dist)

    def build_inter_layer_links(self):
        """Build links between adjacent layers"""
        for layer_idx in range(1, len(self.layers)):
            higher_layer_nodes = list(self.layers[layer_idx].nodes())
            lower_layer_nodes = list(self.layers[layer_idx-1].nodes())

            if not higher_layer_nodes or not lower_layer_nodes:
                continue

            lower_embeddings = np.array([self.node_embeddings[n] for n in lower_layer_nodes])
            nn = NearestNeighbors(n_neighbors=1, metric="cosine")
            nn.fit(lower_embeddings)

            for node_id in higher_layer_nodes:
                if node_id not in self.node_embeddings:
                    continue

                node_emb = self.node_embeddings[node_id]
                distances, indices = nn.kneighbors([node_emb])
                nearest = lower_layer_nodes[indices[0][0]]
                self.inter_layer_links[node_id] = nearest

    def search_layer(self, layer_idx: int, query_embedding: np.ndarray,
                    entry_point: str, k: int = 1) -> List[str]:
        """Search for k nearest neighbors in a specific layer using edge weights"""
        if layer_idx >= len(self.layers) or not self.layers[layer_idx].nodes():
            return []

        if entry_point not in self.node_embeddings:
            return []

        layer = self.layers[layer_idx]
        visited = set([entry_point])
        candidates = [(self.distance(query_embedding, self.node_embeddings[entry_point]), entry_point)]
        results = [(self.distance(query_embedding, self.node_embeddings[entry_point]), entry_point)]

        while candidates:
            current_dist, current = heapq.heappop(candidates)

            if results and current_dist > max(results, key=lambda x: x[0])[0]:
                break

            for neighbor in layer.neighbors(current):
                if neighbor not in visited and neighbor in self.node_embeddings:
                    visited.add(neighbor)
                    dist = self.distance(query_embedding, self.node_embeddings[neighbor])
                    edge_data = layer.get_edge_data(current, neighbor)
                    weight = edge_data.get("weight", 1.0)

                    adjusted_dist = dist * weight
                    if len(results) < k or adjusted_dist < max(results, key=lambda x: x[0])[0]:
                        heapq.heappush(candidates, (adjusted_dist, neighbor))
                        heapq.heappush(results, (adjusted_dist, neighbor))

                        if len(results) > k:
                            results.remove(max(results, key=lambda x: x[0]))

        return [node_id for _, node_id in sorted(results, key=lambda x: x[0])[:k]]

    def hierarchical_search(self, query_embedding: np.ndarray, k: int = 3) -> Dict[int, List[str]]:
        """Perform hierarchical search across all layers"""
        if not self.layers or all(not layer.nodes() for layer in self.layers):
            return {}

        results = {}
        top_layer = len(self.layers) - 1
        while top_layer >= 0 and not self.layers[top_layer].nodes():
            top_layer -= 1

        if top_layer < 0:
            return {}

        entry_point = list(self.layers[top_layer].nodes())[0]

        for layer_idx in range(top_layer, -1, -1):
            if not self.layers[layer_idx].nodes():
                continue

            layer_results = self.search_layer(layer_idx, query_embedding, entry_point, k)
            results[layer_idx] = layer_results

            if layer_results and layer_idx > 0:
                best_node = layer_results[0]
                entry_point = self.inter_layer_links.get(best_node,
                    list(self.layers[layer_idx - 1].nodes())[0] if self.layers[layer_idx - 1].nodes() else entry_point)

        return results

In [ ]:
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
import networkx as nx
from graspologic.partition import hierarchical_leiden
from collections import defaultdict
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import json
import logging

class ArchRAGStore:
    """Enhanced ArchRAG Store with hierarchical communities"""

    def __init__(self, uri: str, username: str, password: str):
        self.driver = GraphDatabase.driver(uri, auth=(username, password))
        self.embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
        self.entity_embeddings = {}
        self.hierarchical_communities = {}
        self.community_summaries = {}
        self.chnsw_index = ArchRAGCHNSW(embedding_dim=384)
        self.max_layers = 3
        self.min_nodes_per_layer = 3
        self.max_layers = 4  # Increase max layers
        self.min_nodes_per_layer = 2  # Reduce minimum nodes per layer
        self.connection_threshold = 0.1  # Threshold for community connections

    def close(self):
        """Close the database connection"""
        self.driver.close()

    def add_triplets(self, entities: List[tuple], relationships: List[tuple], doc_id: str):
        """Add triplets to Neo4j with enhanced attributes"""
        with self.driver.session() as session:
            for entity_name, entity_type, entity_desc, entity_attrs in entities:
                entity_text = f"{entity_name} {entity_type} {entity_desc} {json.dumps(entity_attrs)}"
                embedding = self.embedding_model.encode(entity_text)
                self.entity_embeddings[entity_name] = embedding

                session.run(
                    """
                    MERGE (e:Entity {name: $name})
                    SET e.type = $type, e.description = $desc, e.doc_id = $doc_id, e.attributes = $attrs
                    """,
                    name=entity_name, type=entity_type, desc=entity_desc,
                    doc_id=doc_id, attrs=json.dumps(entity_attrs)
                )

            for source, target, rel, rel_desc, rel_attrs in relationships:
                session.run(
                    """
                    MATCH (source:Entity {name: $source})
                    MATCH (target:Entity {name: $target})
                    MERGE (source)-[r:RELATION {type: $rel}]->(target)
                    SET r.description = $desc, r.doc_id = $doc_id, r.attributes = $attrs
                    """,
                    source=source, target=target, rel=rel, desc=rel_desc,
                    doc_id=doc_id, attrs=json.dumps(rel_attrs)
                )

    def _calculate_community_connection(self, comm1: dict, comm2: dict, graph: nx.Graph) -> float:
        """Calculate connection strength between two communities"""
        members1 = set(comm1["members"])
        members2 = set(comm2["members"])

        # Direct connections between members
        direct_connections = 0
        total_possible = len(members1) * len(members2)

        if total_possible == 0:
            return 0.0

        for m1 in members1:
            for m2 in members2:
                if graph.has_edge(m1, m2):
                    direct_connections += 1

        # Shared members (for overlapping communities)
        shared_members = len(members1.intersection(members2))

        # Calculate connection strength
        connection_strength = (direct_connections / total_possible) + (shared_members / max(len(members1), len(members2)))

        return min(connection_strength, 1.0)

    def compute_similarity_threshold(self, similarities: List[float], percentile: float = 0.7) -> float:
        """Compute dynamic similarity threshold"""
        if not similarities:
            return 0.5
        return np.percentile(similarities, percentile * 100)

    def augment_graph_with_attributes(self, nx_graph: nx.Graph, k: int = 5) -> nx.Graph:
        """Augment graph by connecting entities with similar attributes"""
        augmented_graph = nx_graph.copy()
        nodes = list(nx_graph.nodes())

        if len(nodes) <= 1:
            return augmented_graph

        all_similarities = []
        node_similarities = {}

        for i, node1 in enumerate(nodes):
            if node1 not in self.entity_embeddings:
                continue

            similarities = []
            for j, node2 in enumerate(nodes):
                if i != j and node2 in self.entity_embeddings:
                    sim = cosine_similarity(
                        [self.entity_embeddings[node1]],
                        [self.entity_embeddings[node2]]
                    )[0][0]
                    similarities.append((node2, sim))
                    all_similarities.append(sim)

            node_similarities[node1] = similarities

        threshold = self.compute_similarity_threshold(all_similarities)

        for node1, similarities in node_similarities.items():
            top_k = sorted(similarities, key=lambda x: x[1], reverse=True)[:k]
            for node2, sim in top_k:
                if sim > threshold:
                    weight = 1 - sim
                    augmented_graph.add_edge(node1, node2, weight=weight, similarity=sim)

        return augmented_graph

    def cluster_graph(self, graph: nx.Graph, method: str = "leiden") -> List[Set[str]]:
        """Cluster graph using specified method"""
        if len(graph.nodes()) < 2:
            return [set(graph.nodes())]

        try:
            if method == "leiden":
                clusters = hierarchical_leiden(graph, max_cluster_size=5)
                community_map = defaultdict(set)
                for item in clusters:
                    community_map[item.cluster].add(item.node)
                return list(community_map.values())
            else:
                communities = nx.algorithms.community.greedy_modularity_communities(graph)
                return [set(c) for c in communities]
        except Exception as e:
            logging.error(f"Clustering failed: {e}, falling back to single community")
            return [set(graph.nodes())]

    def generate_attributed_community_summary(self, community: Set[str], llm) -> str:
        """Generate summary for attributed community using LLM"""

        # Check if this is a meta-community (contains other community IDs)
        community_ids = {entity for entity in community if entity.startswith('L')}
        actual_entities = {entity for entity in community if not entity.startswith('L')}

        if community_ids and not actual_entities:
            # This is a pure meta-community - summarize the lower-level communities
            return self._generate_meta_community_summary(community_ids, llm)
        # elif community_ids and actual_entities:
        #     # Mixed community - handle both
        #     return self._generate_mixed_community_summary(community_ids, actual_entities, llm)
        else:
            # Regular entity community - use existing logic
            return self._generate_entity_community_summary(actual_entities, llm)

    def _generate_meta_community_summary(self, community_ids: Set[str], llm) -> str:
        """Generate summary for meta-community containing other communities"""
        sub_summaries = []
        total_members = 0

        for comm_id in community_ids:
            # Find the community data from lower layers
            for layer_data in self.hierarchical_communities.values():
                for comm_data in layer_data:
                    if comm_data['id'] == comm_id:
                        sub_summaries.append(comm_data['summary'])
                        total_members += comm_data['size']
                        break

        if not sub_summaries:
            return f"Meta-community grouping {len(community_ids)} sub-communities"

        combined_text = "\n\n".join(sub_summaries)

        meta_prompt = ChatPromptTemplate.from_messages([
            ("system",
            "You are analyzing a higher-level community that groups several sub-communities. "
            "Create a concise summary that identifies the overarching themes and patterns "
            "across these sub-communities. Focus on what connects them at a higher level."),
            ("human", "Sub-community summaries:\n{summaries}")
        ])

        meta_chain = LLMChain(llm=llm, prompt=meta_prompt)
        try:
            result = meta_chain.invoke({"summaries": combined_text})
            return f"Meta-community ({total_members} total entities): {result['text'].strip()}"
        except Exception as e:
            logging.error(f"Error generating meta-community summary: {e}")
            return f"Meta-community grouping {len(community_ids)} sub-communities with {total_members} total entities"

    def _generate_entity_community_summary(self, actual_entities: Set[str], llm) -> str:
        """Generate summary for regular entity community (existing logic)"""
        # Your existing entity summary logic here
        entity_details = []
        relationship_details = []

        with self.driver.session() as session:
            for entity_name in actual_entities:
                result = session.run(
                    "MATCH (e:Entity {name: $name}) RETURN e.name, e.type, e.description, e.attributes",
                    name=entity_name
                )
                record = result.single()
                if record:
                    entity_details.append({
                        "name": record["e.name"],
                        "type": record["e.type"] or "Unknown",
                        "description": record["e.description"] or "No description",
                        "attributes": json.loads(record["e.attributes"] or "{}")
                    })

        if not entity_details:
            return f"Community with {len(actual_entities)} entities"

        # Rest of your existing entity summary logic...
        summary_text = f"Entities: {json.dumps(entity_details, indent=2)}"

        summary_prompt = ChatPromptTemplate.from_messages([
            ("system", "Create a concise summary of this entity community."),
            ("human", "Community data:\n{community_data}")
        ])

        summary_chain = LLMChain(llm=llm, prompt=summary_prompt)
        try:
            result = summary_chain.invoke({"community_data": summary_text})
            return result["text"].strip()
        except Exception as e:
            logging.error(f"Error generating entity summary: {e}")
            return f"Community with {len(actual_entities)} entities"

    def build_hierarchical_attributed_communities(self, llm):
        """Build hierarchical attributed communities"""
        current_graph = self._create_nx_graph()
        layer = 0
        all_communities = {}

        logging.info(f"Starting hierarchical clustering with {len(current_graph.nodes())} nodes")

        while layer < self.max_layers and len(current_graph.nodes()) >= self.min_nodes_per_layer:
            logging.info(f"Processing layer {layer} with {len(current_graph.nodes())} nodes")

            augmented_graph = self.augment_graph_with_attributes(current_graph)
            communities = self.cluster_graph(augmented_graph)
            logging.info(f"Found {len(communities)} communities in layer {layer}")

            layer_communities = []
            for i, community in enumerate(communities):
                if len(community) == 0:
                    continue

                # DON'T SKIP - Process all communities including meta-communities
                community_id = f"L{layer}_C{i}"
                summary = self.generate_attributed_community_summary(community, llm)

                community_data = {
                    "id": community_id,
                    "layer": layer,
                    "members": list(community),
                    "summary": summary,
                    "size": len(community),
                    "is_meta_community": any(member.startswith('L') for member in community)
                }
                layer_communities.append(community_data)

                try:
                    self.entity_embeddings[community_id] = self.embedding_model.encode(summary)
                except Exception as e:
                    logging.error(f"Error generating embedding for community {community_id}: {e}")
                    self.entity_embeddings[community_id] = np.zeros(384)

            all_communities[layer] = layer_communities
            self.hierarchical_communities[layer] = layer_communities

            # Build next layer graph
            new_graph = nx.Graph()
            for comm_data in layer_communities:
                new_graph.add_node(comm_data["id"])

            # Connect communities that share members or have connections
            for i, comm1 in enumerate(layer_communities):
                for j, comm2 in enumerate(layer_communities):
                    if i < j:
                        connection_strength = self._calculate_community_connection(
                            comm1, comm2, augmented_graph
                        )
                        if connection_strength > 0.1:  # Threshold for connection
                            new_graph.add_edge(comm1["id"], comm2["id"], weight=1-connection_strength)

            current_graph = new_graph
            layer += 1

        logging.info(f"Built {layer} layers of hierarchical communities")
        return all_communities

    def build_chnsw_index(self):
        """Build C-HNSW index from hierarchical communities"""
        logging.info("Building C-HNSW index...")

        # Add all communities to appropriate layers
        for layer, communities in self.hierarchical_communities.items():
            for community in communities:
                community_id = community["id"]
                if community_id in self.entity_embeddings:
                    embedding = self.entity_embeddings[community_id]
                    self.chnsw_index.add_node(
                        community_id,
                        embedding,
                        layer,
                        node_data=community
                    )

        # Add individual entities to layer 0
        for entity_name, embedding in self.entity_embeddings.items():
            if not entity_name.startswith('L'):  # Skip community IDs
                self.chnsw_index.add_node(
                    entity_name,
                    embedding,
                    0,
                    node_data={"type": "entity", "name": entity_name}
                )

        self.chnsw_index.build_intra_layer_links()
        self.chnsw_index.build_inter_layer_links()
        logging.info("C-HNSW index construction completed")

    def _create_nx_graph(self) -> nx.Graph:
        """Create NetworkX graph from Neo4j database"""
        nx_graph = nx.Graph()
        with self.driver.session() as session:
            result = session.run(
                "MATCH (e1)-[r:RELATION]->(e2) RETURN e1.name, r.type, e2.name, r.description, r.attributes"
            )
            for record in result:
                e1, rel, e2, desc, attrs = (
                    record["e1.name"], record["r.type"], record["e2.name"],
                    record["r.description"], record["r.attributes"]
                )
                nx_graph.add_node(e1)
                nx_graph.add_node(e2)
                nx_graph.add_edge(e1, e2, relationship=rel, description=desc, attributes=attrs)
        return nx_graph

    def build_communities(self, llm):
        """Main method to build the complete hagRAG structure"""
        logging.info("Building hagRAG hierarchical attributed communities...")
        self.build_hierarchical_attributed_communities(llm)
        self.build_chnsw_index()
        logging.info("haghRAG community structure completed!")

In [ ]:
#updated cell with llm chain error fixed. cell below will be removed if it works successfully, ok?
import json
import numpy as np
import logging
import re
from typing import Dict, Any, List, Optional
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from langchain_huggingface import HuggingFacePipeline
from tqdm import tqdm
from langchain_core.prompts import ChatPromptTemplate
 #  ADDED - was missing

class ArchRAGQueryEngine:
    """Enhanced Query Engine for ArchRAG"""

    def __init__(self, graph_store, llm, relevance_threshold: float = 0.3):
        self.graph_store = graph_store
        self.llm = llm
        self.embedding_model = graph_store.embedding_model
        self.relevance_threshold = relevance_threshold

    def retrieve_hierarchical_info(self, query: str, k: int = 3) -> Dict[int, List[Dict]]:
        """Retrieve relevant information across all hierarchical layers"""
        query_embedding = self.embedding_model.encode(query)
        search_results = self.graph_store.chnsw_index.hierarchical_search(query_embedding, k)

        hierarchical_info = {}
        for layer, node_ids in search_results.items():
            layer_info = []
            for node_id in node_ids:
                if node_id in self.graph_store.chnsw_index.node_data:
                    node_data = self.graph_store.chnsw_index.node_data[node_id]
                    if node_data.get("type") == "entity":
                        entity_details = self._get_entity_details(node_id)
                        layer_info.append({
                            "id": node_id,
                            "type": "entity",
                            "content": entity_details,
                            "layer": layer
                        })
                    else:
                        layer_info.append({
                            "id": node_id,
                            "type": "community",
                            "content": node_data,
                            "layer": layer
                        })
            hierarchical_info[layer] = layer_info
        return hierarchical_info

    def _get_entity_details(self, entity_name: str) -> Dict:
        """Get detailed information about an entity from Neo4j"""
        with self.graph_store.driver.session() as session:
            result = session.run(
                """
                MATCH (e:Entity {name: $name})
                OPTIONAL MATCH (e)-[r:RELATION]-(connected)
                RETURN e.name, e.type, e.description, e.attributes,
                       collect({relation: r.type, connected_entity: connected.name,
                               relation_desc: r.description}) as relationships
                """,
                name=entity_name
            )
            record = result.single()
            if record:
                return {
                    "name": record["e.name"],
                    "type": record["e.type"] or "Unknown",
                    "description": record["e.description"] or "No description",
                    "attributes": json.loads(record["e.attributes"] or "{}"),
                    "relationships": record["relationships"]
                }
            return {}

    def adaptive_filter_information(self, query: str, hierarchical_info: Dict[int, List[Dict]]) -> List[Dict]:
        """Apply adaptive filtering to retrieved information with layer weighting"""
        analysis_reports = []
        max_layer = max(hierarchical_info.keys(), default=0)

        for layer, layer_info in sorted(hierarchical_info.items(), reverse=True):
            if not layer_info:
                continue

            layer_text = self._format_layer_info(layer_info)
            layer_weight = 1.0 + (layer / max_layer) * 0.5 if max_layer > 0 else 1.0

            filter_prompt = ChatPromptTemplate.from_messages([
                ("system",
                "Analyze the relevance of the provided information for the query. "
                "Respond with ONLY a valid JSON object in this exact format:\n"
                '{"relevance_score": 7.5, "analysis": "brief analysis text", "relevant_content": "most relevant parts"}\n'
                "relevance_score must be a number from 0-10. Do not include any text before or after the JSON."),
                ("human", "Query: {query}\n\nRetrieved Information:\n{info}")
            ])

            filter_chain = LLMChain(llm=self.llm, prompt=filter_prompt)

            response_text = ""
            try:
                response = filter_chain.invoke({"query": query, "info": layer_text})
                response_text = response["text"].strip()
                logging.debug(f"LLM response for layer {layer}: {response_text}")

                json_pattern = r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}'
                matches = re.findall(json_pattern, response_text, re.DOTALL)

                report = None
                for match in matches:
                    try:
                        potential_report = json.loads(match)
                        if isinstance(potential_report, dict) and "relevance_score" in potential_report:
                            report = potential_report
                            break
                    except json.JSONDecodeError:
                        continue

                if not report:
                    relevance_match = re.search(r'"relevance_score":\s*(\d+\.?\d*)', response_text)
                    relevance_score = float(relevance_match.group(1)) if relevance_match else 5.0

                    report = {
                        "relevance_score": relevance_score,
                        "analysis": "Auto-generated analysis due to parsing issues",
                        "relevant_content": layer_text[:500] + "..." if len(layer_text) > 500 else layer_text
                    }

                report["relevance_score"] = float(report.get("relevance_score", 5.0))
                report["analysis"] = str(report.get("analysis", "No analysis available"))
                report["relevant_content"] = str(report.get("relevant_content", layer_text))

                report["relevance_score"] = min(report["relevance_score"] * layer_weight, 10)
                report["layer"] = layer
                report["original_info"] = layer_info
                analysis_reports.append(report)

            except Exception as e:
                analysis_reports.append({
                    "relevance_score": 3.0 * layer_weight,
                    "analysis": f"Processing fallback due to error: {str(e)[:100]}",
                    "relevant_content": layer_text[:500] + "..." if len(layer_text) > 500 else layer_text,
                    "layer": layer,
                    "original_info": layer_info
                })

        analysis_reports = [r for r in analysis_reports if r.get("relevance_score", 0) >= self.relevance_threshold * 10]
        analysis_reports.sort(key=lambda x: x.get("relevance_score", 0), reverse=True)
        return analysis_reports

    def _format_layer_info(self, layer_info: List[Dict]) -> str:
        """Format layer information for LLM processing"""
        formatted_parts = []
        for item in layer_info:
            if item["type"] == "entity":
                content = item["content"]
                formatted_parts.append(
                    f"Entity: {content.get('name', 'Unknown')}\n"
                    f"Type: {content.get('type', 'Unknown')}\n"
                    f"Description: {content.get('description', 'No description')}\n"
                    f"Attributes: {json.dumps(content.get('attributes', {}))}\n"
                    f"Relationships: {json.dumps(content.get('relationships', []))}\n"
                )
            elif item["type"] == "community":
                content = item["content"]
                formatted_parts.append(
                    f"Community {content.get('id', 'Unknown')} (Layer {content.get('layer', 'Unknown')}):\n"
                    f"Summary: {content.get('summary', 'No summary')}\n"
                    f"Size: {content.get('size', 0)} members\n"
                    f"Members: {', '.join(content.get('members', []))}\n"
                )
        return "\n---\n".join(formatted_parts)

    def generate_response(self, query: str, filtered_info: List[Dict]) -> str:
        """Generate final response using filtered information"""
        if not filtered_info:
            return "I couldn't find relevant information to answer your query."

        context_parts = [report.get("relevant_content", "") for report in filtered_info]
        context = "\n\n".join(context_parts)

        response_prompt = ChatPromptTemplate.from_messages([
            ("system",
             "Answer the query based on the provided context. "
             "Cite specific entities or relationships when relevant. "
             "If the context is insufficient, acknowledge the limitations and provide a general answer."),
            ("human", "Question: {query}\n\nContext:\n{context}")
        ])

        response_chain = LLMChain(llm=self.llm, prompt=response_prompt)

        try:
            result = response_chain.invoke({"query": query, "context": context})
            return result["text"].strip()
        except Exception as e:
            logging.error(f"Error generating response: {e}")
            return f"Error generating response: {e}"

    def query(self, query: str, k: int = 3) -> Dict[str, Any]:
        """Main query method for ArchRAG pipeline"""
        logging.info(f"Processing query: {query}")

        hierarchical_info = self.retrieve_hierarchical_info(query, k)
        filtered_info = self.adaptive_filter_information(query, hierarchical_info)
        response = self.generate_response(query, filtered_info)

        return {
            "query": query,
            "response": response,
            "hierarchical_info": hierarchical_info,
            "filtered_info": filtered_info,
            "layers_searched": list(hierarchical_info.keys())
        }

class ArchRAGPipeline:
    """Complete ArchRAG Pipeline"""
    
    def __init__(self, pdf_dir: str, neo4j_uri: str, neo4j_username: str,
                 neo4j_password: str):
        # Set cache dir to a path with enough space
        os.environ["HF_HOME"] = "./workspace/hf_cache"
        os.environ["TRANSFORMERS_CACHE"] = "./workspace/hf_cache"
        
        self.pdf_processor = PDFProcessor(pdf_dir)
        self.graph_store = ArchRAGStore(neo4j_uri, neo4j_username, neo4j_password)
        
        # Initialize Llama 3.1 with 4-bit quantization for better performance
        model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
        
        try:
            # Configure quantization with more specific settings
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
            )
            
            # Load tokenizer first
            tokenizer = AutoTokenizer.from_pretrained(
                model_id, 
                token=os.getenv("HF_TOKEN"),
                padding_side="left"
            )
            
            # Set pad token if it doesn't exist
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
            
            # Load model with minimal device mapping to avoid conflicts
            model = AutoModelForCausalLM.from_pretrained(
                model_id,
                token=os.getenv("HF_TOKEN"),
                quantization_config=quantization_config,
                low_cpu_mem_usage=True,
                trust_remote_code=True,
            )
            
            #  FIXED: Create raw pipeline for TripletExtractor
            raw_pipeline = pipeline(
                "text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=512,
                temperature=0.1,
                do_sample=True,
                top_p=0.95,
                repetition_penalty=1.2,
                return_full_text=False,
                device=model.device if hasattr(model, 'device') else 0
            )
            
            print(f"Model loaded successfully on device: {model.device}")
            
        except Exception as e:
            print(f"Error loading quantized model: {e}")
            print("Falling back to non-quantized model...")
            
            # Fallback: Load without quantization
            tokenizer = AutoTokenizer.from_pretrained(
                model_id, 
                token=os.getenv("HF_TOKEN"),
                padding_side="left"
            )
            
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
            
            model = AutoModelForCausalLM.from_pretrained(
                model_id,
                token=os.getenv("HF_TOKEN"),
                torch_dtype=torch.float16,
                device_map="auto",
                low_cpu_mem_usage=True,
                trust_remote_code=True
            )
            
            #  FIXED: Create raw pipeline for TripletExtractor
            raw_pipeline = pipeline(
                "text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=512,
                temperature=0.1,
                do_sample=True,
                top_p=0.95,
                repetition_penalty=1.2,
                return_full_text=False
            )
        
        #  FIXED: Create LangChain wrapper separately
        hf_llm = HuggingFacePipeline(pipeline=raw_pipeline)
        
        #  FIXED: Pass correct objects to each component
        self.triplet_extractor = TripletExtractor(raw_pipeline)  # Raw pipeline
        self.query_engine = ArchRAGQueryEngine(self.graph_store, hf_llm)  # LangChain wrapper

    def build_knowledge_graph(self, max_docs: Optional[int] = None):
        """Build the complete knowledge graph from PDFs"""
        logging.info("Starting ArchRAG knowledge graph construction...")

        texts = self.pdf_processor.extract_texts()
        if max_docs:
            texts = texts[:max_docs]
        logging.info(f"Extracted text from {len(texts)} documents")

        chunks = self.pdf_processor.create_chunks(texts)
        logging.info(f"Created {len(chunks)} chunks")

        for i, chunk in enumerate(tqdm(chunks, desc="Processing chunks")):
            doc_id = f"doc_{i // 20}_chunk_{i % 20}"
            entities, relationships = self.triplet_extractor.extract_triplets(chunk)
            if entities or relationships:
                self.graph_store.add_triplets(entities, relationships, doc_id)

        #  FIXED: Pass LangChain wrapper to build_communities
        self.graph_store.build_communities(self.query_engine.llm)
        logging.info("ArchRAG knowledge graph construction completed!")

    def query(self, query: str, k: int = 10) -> Dict[str, Any]:
        """Query the ArchRAG system"""
        return self.query_engine.query(query, k)

    def close(self):
        """Clean up resources"""
        self.graph_store.close()

# Utility functions
def analyze_graph_structure(graph_store: ArchRAGStore):
    """Analyze the structure of the built knowledge graph"""
    with graph_store.driver.session() as session:
        entity_count = session.run("MATCH (e:Entity) RETURN count(e) as count").single()["count"]
        rel_count = session.run("MATCH ()-[r:RELATION]->() RETURN count(r) as count").single()["count"]
        entity_types = session.run(
            "MATCH (e:Entity) RETURN e.type, count(e) as count ORDER BY count DESC"
        )

        print(f"\nGraph Structure Analysis:")
        print(f"Total Entities: {entity_count}")
        print(f"Total Relationships: {rel_count}")
        print(f"Entity Types:")
        for record in entity_types:
            print(f"  {record['e.type']}: {record['count']}")

        print(f"\nHierarchical Communities:")
        for layer, communities in graph_store.hierarchical_communities.items():
            print(f"Layer {layer}: {len(communities)} communities")
            for comm in communities:
                print(f"  {comm['id']}: {comm['size']} members")

def export_graph_data(graph_store: ArchRAGStore, output_file: str = "archrag_export.json"):
    """Export graph data to JSON file"""
    export_data = {
        "entities": [],
        "relationships": [],
        "hierarchical_communities": graph_store.hierarchical_communities
    }

    with graph_store.driver.session() as session:
        entities = session.run("MATCH (e:Entity) RETURN e")
        for record in entities:
            entity = dict(record["e"])
            export_data["entities"].append(entity)

        relationships = session.run("MATCH ()-[r:RELATION]->() RETURN r")
        for record in relationships:
            rel = dict(record["r"])
            export_data["relationships"].append(rel)

    with open(output_file, 'w') as f:
        json.dump(export_data, f, indent=2, default=str)

    print(f"Graph data exported to {output_file}")

def main():
    """Main execution function"""
    config = {
        "pdf_dir": "/home/871323/RAG/test/150papers/150papers",
        "neo4j_uri": os.getenv("NEO4J_URI"),
        "neo4j_username": os.getenv("NEO4J_USERNAME", "neo4j"),
        "neo4j_password": os.getenv("NEO4J_PASSWORD"),
    }

    pipeline = ArchRAGPipeline(**config)

    try:
        pipeline.build_knowledge_graph(max_docs=150)
        queries = [
            "What are the main concepts discussed in the documents?",
            "How are different entities related to each other?"
        ]

        print("\n" + "="*50)
        print("QUERYING ARCHRAG SYSTEM")
        print("="*50)

        for query in queries:
            print(f"\nQuery: {query}")
            result = pipeline.query(query)
            print(f"Response: {result['response']}")
            print(f"Layers searched: {result['layers_searched']}")
            print("-" * 30)

        analyze_graph_structure(pipeline.graph_store)
        export_graph_data(pipeline.graph_store)

    except Exception as e:
        logging.error(f"Error in main execution: {e}", exc_info=True)
        raise
    finally:
        pipeline.close()

if __name__ == "__main__":
    main()

In [ ]:
import json
import numpy as np
import logging
import re
from typing import Dict, Any, List, Optional
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline
from langchain_huggingface.chat_models import ChatHuggingFace
from tqdm import tqdm
from transformers import pipeline as hf_pipeline
from langchain_core.prompts import ChatPromptTemplate
class ArchRAGQueryEngine:
    """Enhanced Query Engine for ArchRAG"""

    def __init__(self, graph_store, llm, relevance_threshold: float = 0.3):
        self.graph_store = graph_store
        self.llm = llm
        self.embedding_model = graph_store.embedding_model
        self.relevance_threshold = relevance_threshold

    def retrieve_hierarchical_info(self, query: str, k: int = 3) -> Dict[int, List[Dict]]:
        """Retrieve relevant information across all hierarchical layers"""
        query_embedding = self.embedding_model.encode(query)
        search_results = self.graph_store.chnsw_index.hierarchical_search(query_embedding, k)

        hierarchical_info = {}
        for layer, node_ids in search_results.items():
            layer_info = []
            for node_id in node_ids:
                if node_id in self.graph_store.chnsw_index.node_data:
                    node_data = self.graph_store.chnsw_index.node_data[node_id]
                    if node_data.get("type") == "entity":
                        entity_details = self._get_entity_details(node_id)
                        layer_info.append({
                            "id": node_id,
                            "type": "entity",
                            "content": entity_details,
                            "layer": layer
                        })
                    else:
                        layer_info.append({
                            "id": node_id,
                            "type": "community",
                            "content": node_data,
                            "layer": layer
                        })
            hierarchical_info[layer] = layer_info
        return hierarchical_info

    def _get_entity_details(self, entity_name: str) -> Dict:
        """Get detailed information about an entity from Neo4j"""
        with self.graph_store.driver.session() as session:
            result = session.run(
                """
                MATCH (e:Entity {name: $name})
                OPTIONAL MATCH (e)-[r:RELATION]-(connected)
                RETURN e.name, e.type, e.description, e.attributes,
                       collect({relation: r.type, connected_entity: connected.name,
                               relation_desc: r.description}) as relationships
                """,
                name=entity_name
            )
            record = result.single()
            if record:
                return {
                    "name": record["e.name"],
                    "type": record["e.type"] or "Unknown",
                    "description": record["e.description"] or "No description",
                    "attributes": json.loads(record["e.attributes"] or "{}"),
                    "relationships": record["relationships"]
                }
            return {}

    def adaptive_filter_information(self, query: str, hierarchical_info: Dict[int, List[Dict]]) -> List[Dict]:
        """Apply adaptive filtering to retrieved information with layer weighting"""
        analysis_reports = []
        max_layer = max(hierarchical_info.keys(), default=0)

        for layer, layer_info in sorted(hierarchical_info.items(), reverse=True):
            if not layer_info:
                continue

            layer_text = self._format_layer_info(layer_info)
            layer_weight = 1.0 + (layer / max_layer) * 0.5 if max_layer > 0 else 1.0

            filter_prompt = ChatPromptTemplate.from_messages([
                ("system",
                "Analyze the relevance of the provided information for the query. "
                "Respond with ONLY a valid JSON object in this exact format:\n"
                '{"relevance_score": 7.5, "analysis": "brief analysis text", "relevant_content": "most relevant parts"}\n'
                "relevance_score must be a number from 0-10. Do not include any text before or after the JSON."),
                ("human", "Query: {query}\n\nRetrieved Information:\n{info}")
            ])

            filter_chain = LLMChain(llm=self.llm, prompt=filter_prompt)

            response_text = ""  # Initialize to avoid UnboundLocalError
            try:
                response = filter_chain.invoke({"query": query, "info": layer_text})
                response_text = response["text"].strip()
                logging.debug(f"LLM response for layer {layer}: {response_text}")

                # More robust JSON extraction
                json_pattern = r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}'
                matches = re.findall(json_pattern, response_text, re.DOTALL)

                report = None
                for match in matches:
                    try:
                        potential_report = json.loads(match)
                        # Check if it has the required structure
                        if isinstance(potential_report, dict) and "relevance_score" in potential_report:
                            report = potential_report
                            break
                    except json.JSONDecodeError:
                        continue

                if not report:
                    # Fallback: try to extract values manually
                    relevance_match = re.search(r'"relevance_score":\s*(\d+\.?\d*)', response_text)
                    relevance_score = float(relevance_match.group(1)) if relevance_match else 5.0

                    report = {
                        "relevance_score": relevance_score,
                        "analysis": "Auto-generated analysis due to parsing issues",
                        "relevant_content": layer_text[:500] + "..." if len(layer_text) > 500 else layer_text
                    }

                # Validate and fix the report
                report["relevance_score"] = float(report.get("relevance_score", 5.0))
                report["analysis"] = str(report.get("analysis", "No analysis available"))
                report["relevant_content"] = str(report.get("relevant_content", layer_text))

                # Apply layer weighting
                report["relevance_score"] = min(report["relevance_score"] * layer_weight, 10)
                report["layer"] = layer
                report["original_info"] = layer_info
                analysis_reports.append(report)

            except Exception as e:
                # Robust fallback for any error
                analysis_reports.append({
                    "relevance_score": 3.0 * layer_weight,
                    "analysis": f"Processing fallback due to error: {str(e)[:100]}",
                    "relevant_content": layer_text[:500] + "..." if len(layer_text) > 500 else layer_text,
                    "layer": layer,
                    "original_info": layer_info
                })

        analysis_reports = [r for r in analysis_reports if r.get("relevance_score", 0) >= self.relevance_threshold * 10]
        analysis_reports.sort(key=lambda x: x.get("relevance_score", 0), reverse=True)
        return analysis_reports

    def _format_layer_info(self, layer_info: List[Dict]) -> str:
        """Format layer information for LLM processing"""
        formatted_parts = []
        for item in layer_info:
            if item["type"] == "entity":
                content = item["content"]
                formatted_parts.append(
                    f"Entity: {content.get('name', 'Unknown')}\n"
                    f"Type: {content.get('type', 'Unknown')}\n"
                    f"Description: {content.get('description', 'No description')}\n"
                    f"Attributes: {json.dumps(content.get('attributes', {}))}\n"
                    f"Relationships: {json.dumps(content.get('relationships', []))}\n"
                )
            elif item["type"] == "community":
                content = item["content"]
                formatted_parts.append(
                    f"Community {content.get('id', 'Unknown')} (Layer {content.get('layer', 'Unknown')}):\n"
                    f"Summary: {content.get('summary', 'No summary')}\n"
                    f"Size: {content.get('size', 0)} members\n"
                    f"Members: {', '.join(content.get('members', []))}\n"
                )
        return "\n---\n".join(formatted_parts)

    def generate_response(self, query: str, filtered_info: List[Dict]) -> str:
        """Generate final response using filtered information"""
        if not filtered_info:
            return "I couldn't find relevant information to answer your query."

        context_parts = [report.get("relevant_content", "") for report in filtered_info]
        context = "\n\n".join(context_parts)

        response_prompt = ChatPromptTemplate.from_messages([
            ("system",
             "Answer the query based on the provided context. "
             "Cite specific entities or relationships when relevant. "
             "If the context is insufficient, acknowledge the limitations and provide a general answer."),
            ("human", "Question: {query}\n\nContext:\n{context}")
        ])

        response_chain = LLMChain(llm=self.llm, prompt=response_prompt)

        try:
            result = response_chain.invoke({"query": query, "context": context})
            return result["text"].strip()
        except Exception as e:
            logging.error(f"Error generating response: {e}")
            return f"Error generating response: {e}"

    def query(self, query: str, k: int = 3) -> Dict[str, Any]:
        """Main query method for ArchRAG pipeline"""
        logging.info(f"Processing query: {query}")

        hierarchical_info = self.retrieve_hierarchical_info(query, k)
        filtered_info = self.adaptive_filter_information(query, hierarchical_info)
        response = self.generate_response(query, filtered_info)

        return {
            "query": query,
            "response": response,
            "hierarchical_info": hierarchical_info,
            "filtered_info": filtered_info,
            "layers_searched": list(hierarchical_info.keys())
        }

class ArchRAGPipeline:
    """Complete ArchRAG Pipeline"""
    
    def __init__(self, pdf_dir: str, neo4j_uri: str, neo4j_username: str,
                 neo4j_password: str):
        # Set cache dir to a path with enough space
        os.environ["HF_HOME"] = "./workspace/hf_cache"
        os.environ["TRANSFORMERS_CACHE"] = "./workspace/hf_cache"
        
        self.pdf_processor = PDFProcessor(pdf_dir)
        self.graph_store = ArchRAGStore(neo4j_uri, neo4j_username, neo4j_password)
        
        # Initialize Llama 3.1 with 4-bit quantization for better performance
        model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
        
        try:
            # Configure quantization with more specific settings
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,  # Changed from bfloat16
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
            )
            
            # Load tokenizer first
            tokenizer = AutoTokenizer.from_pretrained(
                model_id, 
                token=os.getenv("HF_TOKEN"),
                padding_side="left"
            )
            
            # Set pad token if it doesn't exist
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
            
            # Load model with minimal device mapping to avoid conflicts
            model = AutoModelForCausalLM.from_pretrained(
                model_id,
                token=os.getenv("HF_TOKEN"),
                quantization_config=quantization_config,
                low_cpu_mem_usage=True,
                trust_remote_code=True,
                # Remove device_map and torch_dtype to avoid conflicts with quantization
            )
            
            # Create pipeline with explicit device handling
            pipe = pipeline(
                "text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=512,
                temperature=0.1,
                do_sample=True,
                top_p=0.95,
                repetition_penalty=1.2,
                return_full_text=False,  # Only return generated text
                device=model.device if hasattr(model, 'device') else 0  # Use model's device
            )
            
            print(f"Model loaded successfully on device: {model.device}")
            
        except Exception as e:
            print(f"Error loading quantized model: {e}")
            print("Falling back to non-quantized model...")
            
            # Fallback: Load without quantization
            tokenizer = AutoTokenizer.from_pretrained(
                model_id, 
                token=os.getenv("HF_TOKEN"),
                padding_side="left"
            )
            
            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token
            
            model = AutoModelForCausalLM.from_pretrained(
                model_id,
                token=os.getenv("HF_TOKEN"),
                torch_dtype=torch.float16,
                device_map="auto",
                low_cpu_mem_usage=True,
                trust_remote_code=True
            )
            
            pipe = hf_pipeline(
                "text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=512,
                temperature=0.1,
                do_sample=True,
                top_p=0.95,
                repetition_penalty=1.2,
                return_full_text=False
            )
        
        hf_llm = HuggingFacePipeline(pipeline=pipe)
        self.llm = ChatHuggingFace(llm=hf_llm)  # Wraps for chat template compatibility
        self.triplet_extractor = TripletExtractor(pipe)
        self.query_engine = ArchRAGQueryEngine(self.graph_store, self.llm)

    def build_knowledge_graph(self, max_docs: Optional[int] = None):
        """Build the complete knowledge graph from PDFs"""
        logging.info("Starting ArchRAG knowledge graph construction...")

        texts = self.pdf_processor.extract_texts()
        if max_docs:
            texts = texts[:max_docs]
        logging.info(f"Extracted text from {len(texts)} documents")

        chunks = self.pdf_processor.create_chunks(texts)
        logging.info(f"Created {len(chunks)} chunks")

        for i, chunk in enumerate(tqdm(chunks, desc="Processing chunks")):
            doc_id = f"doc_{i // 10}_chunk_{i % 10}"
            entities, relationships = self.triplet_extractor.extract_triplets(chunk)
            if entities or relationships:
                self.graph_store.add_triplets(entities, relationships, doc_id)

        self.graph_store.build_communities(self.llm)
        logging.info("ArchRAG knowledge graph construction completed!")

    def query(self, query: str, k: int = 10) -> Dict[str, Any]:
        """Query the ArchRAG system"""
        return self.query_engine.query(query, k)

    def close(self):
        """Clean up resources"""
        self.graph_store.close()

# Utility functions
def analyze_graph_structure(graph_store: ArchRAGStore):
    """Analyze the structure of the built knowledge graph"""
    with graph_store.driver.session() as session:
        entity_count = session.run("MATCH (e:Entity) RETURN count(e) as count").single()["count"]
        rel_count = session.run("MATCH ()-[r:RELATION]->() RETURN count(r) as count").single()["count"]
        entity_types = session.run(
            "MATCH (e:Entity) RETURN e.type, count(e) as count ORDER BY count DESC"
        )

        print(f"\nGraph Structure Analysis:")
        print(f"Total Entities: {entity_count}")
        print(f"Total Relationships: {rel_count}")
        print(f"Entity Types:")
        for record in entity_types:
            print(f"  {record['e.type']}: {record['count']}")

        print(f"\nHierarchical Communities:")
        for layer, communities in graph_store.hierarchical_communities.items():
            print(f"Layer {layer}: {len(communities)} communities")
            for comm in communities:
                print(f"  {comm['id']}: {comm['size']} members")

def export_graph_data(graph_store: ArchRAGStore, output_file: str = "archrag_export.json"):
    """Export graph data to JSON file"""
    export_data = {
        "entities": [],
        "relationships": [],
        "hierarchical_communities": graph_store.hierarchical_communities
    }

    with graph_store.driver.session() as session:
        entities = session.run("MATCH (e:Entity) RETURN e")
        for record in entities:
            entity = dict(record["e"])
            export_data["entities"].append(entity)

        relationships = session.run("MATCH ()-[r:RELATION]->() RETURN r")
        for record in relationships:
            rel = dict(record["r"])
            export_data["relationships"].append(rel)

    with open(output_file, 'w') as f:
        json.dump(export_data, f, indent=2, default=str)

    print(f"Graph data exported to {output_file}")

def main():
    """Main execution function"""
    config = {
        "pdf_dir": "/home/871323/RAG/test/150papers/150papers",
        "neo4j_uri": os.getenv("NEO4J_URI"),
        "neo4j_username": os.getenv("NEO4J_USERNAME", "neo4j"),
        "neo4j_password": os.getenv("NEO4J_PASSWORD"),
    }

    pipeline = ArchRAGPipeline(**config)

    try:
        pipeline.build_knowledge_graph(max_docs=150)
        queries = [
            "What are the main concepts discussed in the documents?",
            "How are different entities related to each other?"
        ]

        print("\n" + "="*50)
        print("QUERYING ARCHRAG SYSTEM")
        print("="*50)

        for query in queries:
            print(f"\nQuery: {query}")
            result = pipeline.query(query)
            print(f"Response: {result['response']}")
            print(f"Layers searched: {result['layers_searched']}")
            print("-" * 30)

        analyze_graph_structure(pipeline.graph_store)
        export_graph_data(pipeline.graph_store)

    except Exception as e:
        logging.error(f"Error in main execution: {e}", exc_info=True)
        raise
    finally:
        pipeline.close()

if __name__ == "__main__":
    main()#skip this cell or delete entirely.

INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: all-MiniLM-L6-v2
/home/871323/.local/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Error loading quantized model: No GPU found. A GPU is needed for quantization.
Falling back to non-quantized model...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

INFO:root:Starting ArchRAG knowledge graph construction...
Processing 24107659.pdf:  27%|██▋       | 6/22 [00:01<00:04,  3.88it/s]


Processing 18445730.pdf:  89%|████████▉ | 34/38 [00:04<00:00,  7.67it/s]


...

ERROR:root:Error extracting triplets: 'ChatHuggingFace' object is not callable, Text: Because women with a history of
2004 ADA position statement on gesta- rateofconversiontotype2diabete...
ERROR:root:Error extracting triplets: 'ChatHuggingFace' object is not callable, Text: continuously increased as a function of the National Diabetes Education Pro- stylechangeswithgoalssi...
ERROR:root:Error extracting triplets: 'ChatHuggingFace' object is not callable, Text: PositionStatement
Table6—Therapiesproveneffectiveindiabetespreventiontrials
Mean Conversionin
age Du...
ERROR:root:Error extracting triplets: 'ChatHuggingFace' object is not callable, Text: DREAM(15) 5,269 IGTorIFG 55 3.0 Rosiglitazone(8mg) 9 0.40(0.35–0.46)
*Numberofparticipantsintheindic...
ERROR:root:Error extracting triplets: 'ChatHuggingFace' object is not callable, Text: sistence of effect in some studies led the ical care from a physician-coordinated ditions.
paneltono...
ERROR:root:Error extracting triplets: 'ChatHugging

ERROR:root:Error extracting triplets: 'ChatHuggingFace' object is not callable, Text: b.Therapyfortype2diabetes. TheADA drate or low-fat calorie restricted diets dayorlessforadultmen).(E...
ERROR:root:Error extracting triplets: 'ChatHuggingFace' object is not callable, Text: tion with lifestyle changes (MNT and tionareimportantcomponentsofweight strated and, therefore, cann...
ERROR:root:Error extracting triplets: 'ChatHuggingFace' object is not callable, Text: notbeingmet. physicalactivity(150min/week),with the evidence regarding nutrition in pre-
DIABETESCAR...
ERROR:root:Error extracting triplets: 'ChatHuggingFace' object is not callable, Text: StandardsofMedicalCare
venting and controlling diabetes and its (10).LookAHEAD(ActionforHealthin sou...
ERROR:root:Error extracting triplets: 'ChatHuggingFace' object is not callable, Text: effortthatincludestheactiveinvolvement nificantreductionofA1C,andreduction The best mix of carbohydr...
ERROR:root:Error extracting triplets: 'ChatHugging

ERROR:root:Error extracting triplets: 'ChatHuggingFace' object is not callable, Text: andmortalityinindividualswithdiabetes ACE inhibitor reduced CVD outcomes 2.Dyslipidemia/lipidmanagem...
ERROR:root:Error extracting triplets: 'ChatHuggingFace' object is not callable, Text: (cid:4)140mmHginpatientswithtype2dia- tinct advantages of RAS inhibitors on mg/dl), lipid assessment...
ERROR:root:Error extracting triplets: 'ChatHuggingFace' object is not callable, Text: StandardsofMedicalCare
Table10—Reductionin10-yearriskofmajorCVDendpoints(CHDdeath/non-fatalMI)inmajo...
ERROR:root:Error extracting triplets: 'ChatHuggingFace' object is not callable, Text: Studieswereofdifferinglengths(3.3–5.4years)andusedsomewhatdifferentoutcomes,butallreportedratesofCVD...
ERROR:root:Error extracting triplets: 'ChatHuggingFace' object is not callable, Text: lipidlevels,fordiabeticpatients: ● Iftargetsarenotreachedonmaximally clearlyseenindiabeticsubjectswi...
ERROR:root:Error extracting triplets: 'ChatHugging

ERROR:root:Error extracting triplets: 'ChatHuggingFace' object is not callable, Text: toryofthediseaseprocessbutmayhavea
atic treatments are available for some Gastrointestinal neuropath...
ERROR:root:Error extracting triplets: 'ChatHuggingFace' object is not callable, Text: if symptoms are suggestive, but test re- amination should include inspection,
plantar aspect of both...
ERROR:root:Error extracting triplets: 'ChatHuggingFace' object is not callable, Text: The symptoms and signs of autonomic tionand/orretrogradeejaculation.Evalu- toallpatientswithdiabetes...
ERROR:root:Error extracting triplets: 'ChatHuggingFace' object is not callable, Text: hypoglycemicautonomicfailure. The first step in management of patients care specialists for ongoing ...
ERROR:root:Error extracting triplets: 'ChatHuggingFace' object is not callable, Text: PositionStatement
ankle-brachialindex(ABI),asmanypa- ropathy.Theclinicalexaminationtoiden- physical ...
ERROR:root:Error extracting triplets: 'ChatHugging

ERROR:root:Error extracting triplets: 'ChatHuggingFace' object is not callable, Text: PositionStatement
glucose(cid:4)100.8mg/dl(5.6mmol/l)than glucose80–110mg/dl[4.4–6.1mmol/l]) 2.Glyce...
ERROR:root:Error extracting triplets: 'ChatHuggingFace' object is not callable, Text: glycemiccontrolintheoutpatientsetting, vivalratesoccurringinpatientsachieving severehypoglycemia(blo...
ERROR:root:Error extracting triplets: 'ChatHuggingFace' object is not callable, Text: admission. For patients in general medical-
thosepatientswhoweretreatedforlonger
In the initial Diab...
ERROR:root:Error extracting triplets: 'ChatHuggingFace' object is not callable, Text: intensive insulin intervention arm was mia, defined as a blood glucose (cid:4)40 getssimilartothoseo...
ERROR:root:Error extracting triplets: 'ChatHuggingFace' object is not callable, Text: showareductioninmortalityintheinter- pear reasonable, if they can be safely
volving 29 studies and o...
ERROR:root:Error extracting triplets: 'ChatHugging

In [ ]:
# ============================================================================
# CELL 5: BUILD COMPLETE HA-GRAG HIERARCHICAL STRUCTURE
# This properly builds layers with intra/inter-layer links
# ============================================================================

import os
import logging
from tqdm import tqdm
import nltk
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline
from langchain_huggingface.chat_models import ChatHuggingFace
from transformers import pipeline as hf_pipeline
import json

logging.basicConfig(level=logging.INFO)

# Download NLTK data
try:
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)
except:
    pass

# Configuration
config = {
    "pdf_dir": "/home/871323/RAG/test/150papers/150papers",
    "neo4j_uri": os.getenv("NEO4J_URI"),
    "neo4j_username": os.getenv("NEO4J_USERNAME", "neo4j"),
    "neo4j_password": os.getenv("NEO4J_PASSWORD"),
}

print("="*80)
print("BUILDING COMPLETE HA-GRAG HIERARCHICAL STRUCTURE")
print("="*80)

# ============================================================================
# STEP 1: INITIALIZE GRAPH STORE
# ============================================================================

print("\n[Step 1/5] Initializing graph store...")
graph_store = ArchRAGStore(
    uri=config["neo4j_uri"],
    username=config["neo4j_username"],
    password=config["neo4j_password"]
)
print(" Graph store initialized")

# Check if graph exists
with graph_store.driver.session() as session:
    result = session.run("MATCH (e:Entity) RETURN count(e) as count")
    entity_count = result.single()["count"]

print(f" Found {entity_count} base entities in Neo4j")

if entity_count == 0:
    print("\n  ERROR: No entities found in database!")
    print("  You need to build the knowledge graph first.")
    print("  Run: pipeline.build_knowledge_graph(max_docs=150)")
    raise ValueError("No entities in database")

# ============================================================================
# STEP 2: LOAD ENTITY EMBEDDINGS
# ============================================================================

print("\n[Step 2/5] Loading entity embeddings...")

with graph_store.driver.session() as session:
    result = session.run("""
        MATCH (e:Entity) 
        WHERE NOT e.name STARTS WITH 'L'
        RETURN e.name as name, e.type as type, e.description as desc, e.attributes as attrs
    """)
    
    count = 0
    for record in result:
        entity_name = record['name']
        if entity_name and entity_name not in graph_store.entity_embeddings:
            entity_text = f"{entity_name} {record['type']} {record['desc']} {record['attrs']}"
            graph_store.entity_embeddings[entity_name] = graph_store.embedding_model.encode(entity_text)
            count += 1
            
            if count % 100 == 0:
                print(f"  Loaded {count} embeddings...")

print(f" Loaded {len(graph_store.entity_embeddings)} entity embeddings")

# ============================================================================
# STEP 3: INITIALIZE LLM
# ============================================================================
# ============================================================================
# STEP 3: INITIALIZE LLM (WITH AUTOMATIC FALLBACK)
# ============================================================================

print("\n[Step 3/5] Initializing LLM...")

llm = None
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

os.environ["HF_HOME"] = "./workspace/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "./workspace/hf_cache"

try:
    # Try with quantization first
    from bitsandbytes import BitsAndBytesConfig
    
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
    
    tokenizer = AutoTokenizer.from_pretrained(
        model_id, 
        token=os.getenv("HF_TOKEN"),
        padding_side="left"
    )
    
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        token=os.getenv("HF_TOKEN"),
        quantization_config=quantization_config,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )
    
    print(" Loaded with 4-bit quantization")
    
except ImportError:
    print("  bitsandbytes not available, using float16 instead...")
    
    tokenizer = AutoTokenizer.from_pretrained(
        model_id, 
        token=os.getenv("HF_TOKEN"),
        padding_side="left"
    )
    
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        token=os.getenv("HF_TOKEN"),
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
        trust_remote_code=True
    )
    
    print(" Loaded with float16 (no quantization)")

# Create pipeline
pipe = hf_pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,
    do_sample=True,
    top_p=0.95,
    repetition_penalty=1.2,
    return_full_text=False
)

hf_llm = HuggingFacePipeline(pipeline=pipe)
llm = ChatHuggingFace(llm=hf_llm)

print(" LLM ready")

# ============================================================================
# STEP 4: BUILD HIERARCHICAL COMMUNITIES (Layer by Layer)
# ============================================================================

print("\n[Step 4/5] Building hierarchical communities...")
print("This creates Layer 0  Layer 1  Layer 2 with proper structure\n")

# Check if communities already exist
check_existing = False
if hasattr(graph_store, 'hierarchical_communities') and graph_store.hierarchical_communities:
    print("Found existing communities in memory:")
    for layer, comms in graph_store.hierarchical_communities.items():
        print(f"  Layer {layer}: {len(comms)} communities")
    
    response = input("\nRebuild communities? (y/n): ").strip().lower()
    check_existing = (response != 'y')

if not check_existing:
    print("\nBuilding hierarchical communities from scratch...")
    print("This will take 5-10 minutes...\n")
    
    # Build communities using your existing method
    graph_store.build_communities(llm)
    
    print("\n Hierarchical communities built successfully!")
else:
    print(" Using existing communities")

# Verify communities were built
if not graph_store.hierarchical_communities:
    print("\n ERROR: No communities were built!")
    raise ValueError("Community building failed")

print("\nHierarchical structure:")
for layer, comms in graph_store.hierarchical_communities.items():
    print(f"  Layer {layer}: {len(comms)} communities")
    
    # Show sample community
    if comms:
        sample = comms[0]
        print(f"    Example: {sample['id']} with {sample['size']} members")

# ============================================================================
# STEP 5: BUILD C-HNSW INDEX (Intra & Inter-layer Links)
# ============================================================================

print("\n[Step 5/5] Building C-HNSW index with intra/inter-layer links...")

# Check if index already exists
if hasattr(graph_store, 'chnsw_index') and len(graph_store.chnsw_index.layers) > 0:
    print("Found existing C-HNSW index:")
    for layer_idx, layer in enumerate(graph_store.chnsw_index.layers):
        print(f"  Layer {layer_idx}: {len(layer.nodes())} nodes, {len(layer.edges())} intra-layer links")
    print(f"  Inter-layer links: {len(graph_store.chnsw_index.inter_layer_links)}")
    
    response = input("\nRebuild C-HNSW index? (y/n): ").strip().lower()
    if response == 'y':
        print("\nRebuilding C-HNSW index...")
        graph_store.build_chnsw_index()
    else:
        print(" Using existing C-HNSW index")
else:
    print("\nBuilding C-HNSW index from scratch...")
    graph_store.build_chnsw_index()

# Verify C-HNSW was built
if not graph_store.chnsw_index.layers:
    print("\n ERROR: C-HNSW index was not built!")
    raise ValueError("C-HNSW index building failed")

print("\n C-HNSW index built successfully!")

# ============================================================================
# VERIFICATION AND SUMMARY
# ============================================================================

print("\n" + "="*80)
print("HA-GRAG HIERARCHICAL STRUCTURE - COMPLETE")
print("="*80)

# Count base entities
print("\n1. BASE ENTITIES (Layer 0 - Yellow):")
base_entity_count = sum(1 for name in graph_store.entity_embeddings.keys() 
                        if not name.startswith('L'))
print(f"   Total base entities: {base_entity_count}")

# Hierarchical layers
print("\n2. HIERARCHICAL LAYERS:")
for layer, comms in sorted(graph_store.hierarchical_communities.items()):
    if layer == 0:
        layer_name = "Layer 0 - Entities (Fine-grained) [Yellow]"
    elif layer == 1:
        layer_name = "Layer 1 - Sub-topics (Medium-grained) [Blue]"
    else:
        layer_name = f"Layer {layer} - Main Topics (Coarse-grained) [Red]"
    
    total_members = sum(comm['size'] for comm in comms)
    print(f"   {layer_name}")
    print(f"      Communities: {len(comms)}")
    print(f"      Total members: {total_members}")

# C-HNSW structure
print("\n3. C-HNSW INDEX STRUCTURE:")
for layer_idx, layer_graph in enumerate(graph_store.chnsw_index.layers):
    intra_links = len(layer_graph.edges())
    nodes = len(layer_graph.nodes())
    
    # Calculate avg degree for intra-layer connectivity
    if nodes > 0:
        avg_degree = (2 * intra_links) / nodes
    else:
        avg_degree = 0
    
    print(f"   Layer {layer_idx}:")
    print(f"      Nodes: {nodes}")
    print(f"      INTRA-layer links (green): {intra_links}")
    print(f"      Avg degree: {avg_degree:.2f}")

inter_links = len(graph_store.chnsw_index.inter_layer_links)
print(f"\n   INTER-layer links (purple): {inter_links}")
print(f"      (Connecting adjacent layers for hierarchical search)")

# Link density
print("\n4. LINK DENSITY:")
total_nodes = sum(len(layer.nodes()) for layer in graph_store.chnsw_index.layers)
total_intra_links = sum(len(layer.edges()) for layer in graph_store.chnsw_index.layers)
total_inter_links = len(graph_store.chnsw_index.inter_layer_links)
total_links = total_intra_links + total_inter_links

if total_nodes > 0:
    avg_links_per_node = total_links / total_nodes
    print(f"   Total nodes: {total_nodes}")
    print(f"   Total links: {total_links}")
    print(f"      - Intra-layer: {total_intra_links} ({100*total_intra_links/total_links:.1f}%)")
    print(f"      - Inter-layer: {total_inter_links} ({100*total_inter_links/total_links:.1f}%)")
    print(f"   Avg links per node: {avg_links_per_node:.2f}")

print("\n" + "="*80)
print(" HIERARCHICAL STRUCTURE READY FOR EVALUATION!")
print("="*80)
print("\nYou can now run:")
print("  1. GraphQualityEvaluator (with hierarchical metrics)")
print("  2. Hierarchical visualization")
print("  3. Clustering algorithm comparison")

In [ ]:
# ============================================================================
# CELL 6: ENHANCED GRAPH QUALITY EVALUATOR
# Now includes intra-layer and inter-layer link evaluation
# ============================================================================

import pandas as pd
import numpy as np
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from scipy.stats import entropy
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

class EnhancedGraphQualityEvaluator:
    """Evaluate quality of HA-GRAG graph including hierarchical links"""
    
    def __init__(self, graph_store):
        self.graph_store = graph_store
        self.metrics = {}
        
    def evaluate_all(self):
        """Run all evaluation metrics on your existing graph"""
        print("="*80)
        print("EVALUATING HA-GRAG HIERARCHICAL STRUCTURE")
        print("="*80)
        
        # 1. Structural Metrics
        print("\n[1/5] Evaluating structural quality...")
        structural = self._evaluate_structural()
        
        # 2. Semantic Metrics
        print("[2/5] Evaluating semantic quality...")
        semantic = self._evaluate_semantic()
        
        # 3. Hierarchical Metrics
        print("[3/5] Evaluating hierarchical quality...")
        hierarchical = self._evaluate_hierarchical()
        
        # 4. C-HNSW Link Metrics (NEW!)
        print("[4/5] Evaluating intra/inter-layer links...")
        chnsw = self._evaluate_chnsw_links()
        
        # 5. Information Metrics
        print("[5/5] Evaluating information content...")
        information = self._evaluate_information()
        
        # Combine all metrics
        all_metrics = {
            **{f"Structural_{k}": v for k, v in structural.items()},
            **{f"Semantic_{k}": v for k, v in semantic.items()},
            **{f"Hierarchical_{k}": v for k, v in hierarchical.items()},
            **{f"CHNSW_{k}": v for k, v in chnsw.items()},
            **{f"Information_{k}": v for k, v in information.items()}
        }
        
        # Create DataFrame
        results_df = pd.DataFrame([
            {"Category": k.split('_')[0], "Metric": '_'.join(k.split('_')[1:]), "Value": v}
            for k, v in all_metrics.items()
        ])
        
        print("\n" + "="*80)
        print("EVALUATION COMPLETE!")
        print("="*80)
        
        return results_df
    
    def _evaluate_structural(self):
        """Evaluate structural properties using existing NetworkX graph"""
        metrics = {}
        
        nx_graph = self.graph_store._create_nx_graph()
        
        metrics['num_nodes'] = nx_graph.number_of_nodes()
        metrics['num_edges'] = nx_graph.number_of_edges()
        metrics['density'] = nx.density(nx_graph)
        metrics['avg_clustering_coefficient'] = nx.average_clustering(nx_graph)
        metrics['transitivity'] = nx.transitivity(nx_graph)
        
        degrees = [d for n, d in nx_graph.degree()]
        metrics['avg_degree'] = np.mean(degrees) if degrees else 0
        metrics['std_degree'] = np.std(degrees) if degrees else 0
        
        if nx.is_connected(nx_graph):
            metrics['diameter'] = nx.diameter(nx_graph)
            metrics['avg_shortest_path'] = nx.average_shortest_path_length(nx_graph)
            metrics['num_connected_components'] = 1
        else:
            metrics['num_connected_components'] = nx.number_connected_components(nx_graph)
            largest_cc = max(nx.connected_components(nx_graph), key=len)
            subgraph = nx_graph.subgraph(largest_cc)
            metrics['diameter'] = nx.diameter(subgraph)
            metrics['avg_shortest_path'] = nx.average_shortest_path_length(subgraph)
        
        degree_centrality = nx.degree_centrality(nx_graph)
        metrics['avg_degree_centrality'] = np.mean(list(degree_centrality.values()))
        
        if hasattr(self.graph_store, 'hierarchical_communities') and 0 in self.graph_store.hierarchical_communities:
            layer_0_communities = self.graph_store.hierarchical_communities[0]
            partition_sets = [set(comm['members']) for comm in layer_0_communities]
            try:
                metrics['modularity'] = nx.algorithms.community.modularity(nx_graph, partition_sets)
            except:
                metrics['modularity'] = 0.0
        else:
            metrics['modularity'] = 0.0
        
        return metrics
    
    def _evaluate_semantic(self):
        """Evaluate semantic quality using existing embeddings"""
        metrics = {}
        
        with self.graph_store.driver.session() as session:
            result = session.run("MATCH (e:Entity) RETURN e.type as type, count(e) as count")
            type_counts = {record['type']: record['count'] for record in result if record['type']}
        
        if type_counts:
            total = sum(type_counts.values())
            type_probs = np.array([count / total for count in type_counts.values()])
            metrics['entity_type_diversity'] = entropy(type_probs)
            metrics['num_entity_types'] = len(type_counts)
        else:
            metrics['entity_type_diversity'] = 0.0
            metrics['num_entity_types'] = 0
        
        with self.graph_store.driver.session() as session:
            result = session.run("MATCH ()-[r:RELATION]->() RETURN r.type as type, count(r) as count")
            rel_counts = {record['type']: record['count'] for record in result if record['type']}
        
        if rel_counts:
            total = sum(rel_counts.values())
            rel_probs = np.array([count / total for count in rel_counts.values()])
            metrics['relation_type_diversity'] = entropy(rel_probs)
            metrics['num_relation_types'] = len(rel_counts)
        else:
            metrics['relation_type_diversity'] = 0.0
            metrics['num_relation_types'] = 0
        
        if hasattr(self.graph_store, 'entity_embeddings') and self.graph_store.entity_embeddings:
            entity_embeddings_list = []
            entity_labels = []
            
            with self.graph_store.driver.session() as session:
                for entity_name, embedding in self.graph_store.entity_embeddings.items():
                    if not entity_name.startswith('L'):
                        result = session.run(
                            "MATCH (e:Entity {name: $name}) RETURN e.type as type",
                            name=entity_name
                        )
                        record = result.single()
                        if record and record['type']:
                            entity_embeddings_list.append(embedding)
                            entity_labels.append(record['type'])
            
            if len(entity_embeddings_list) > 10:
                embeddings_array = np.array(entity_embeddings_list)
                unique_types = list(set(entity_labels))
                numeric_labels = [unique_types.index(label) for label in entity_labels]
                
                try:
                    metrics['silhouette_score'] = silhouette_score(embeddings_array, numeric_labels)
                    metrics['davies_bouldin_score'] = davies_bouldin_score(embeddings_array, numeric_labels)
                    metrics['calinski_harabasz_score'] = calinski_harabasz_score(embeddings_array, numeric_labels)
                except:
                    pass
                
                intra_sims = []
                inter_sims = []
                
                for i in range(min(len(entity_embeddings_list), 100)):
                    for j in range(i + 1, min(len(entity_embeddings_list), 100)):
                        sim = cosine_similarity([entity_embeddings_list[i]], [entity_embeddings_list[j]])[0][0]
                        if entity_labels[i] == entity_labels[j]:
                            intra_sims.append(sim)
                        else:
                            inter_sims.append(sim)
                
                if intra_sims and inter_sims:
                    metrics['avg_intra_type_similarity'] = np.mean(intra_sims)
                    metrics['avg_inter_type_similarity'] = np.mean(inter_sims)
                    metrics['semantic_separation_ratio'] = np.mean(intra_sims) / (np.mean(inter_sims) + 1e-10)
        
        with self.graph_store.driver.session() as session:
            result = session.run("""
                MATCH (e:Entity)
                WITH count(e) as total,
                     count(CASE WHEN e.attributes IS NOT NULL AND e.attributes <> '{}' THEN 1 END) as with_attrs
                RETURN toFloat(with_attrs) / total as coverage
            """)
            record = result.single()
            if record:
                metrics['attribute_coverage'] = record['coverage']
        
        return metrics
    
    def _evaluate_hierarchical(self):
        """Evaluate hierarchical structure using your existing communities"""
        metrics = {}
        
        if not hasattr(self.graph_store, 'hierarchical_communities'):
            return metrics
        
        hierarchical_communities = self.graph_store.hierarchical_communities
        
        metrics['num_layers'] = len(hierarchical_communities)
        
        for layer, communities in hierarchical_communities.items():
            metrics[f'layer_{layer}_num_communities'] = len(communities)
            
            if communities:
                sizes = [comm['size'] for comm in communities]
                metrics[f'layer_{layer}_avg_size'] = np.mean(sizes)
                metrics[f'layer_{layer}_std_size'] = np.std(sizes)
                metrics[f'layer_{layer}_min_size'] = np.min(sizes)
                metrics[f'layer_{layer}_max_size'] = np.max(sizes)
        
        if hierarchical_communities:
            layer_0_nodes = sum(comm['size'] for comm in hierarchical_communities.get(0, []))
            top_layer = max(hierarchical_communities.keys())
            top_layer_nodes = len(hierarchical_communities[top_layer])
            
            if layer_0_nodes > 0:
                metrics['compression_ratio'] = top_layer_nodes / layer_0_nodes
        
        if 0 in hierarchical_communities:
            nx_graph = self.graph_store._create_nx_graph()
            layer_0_members = set()
            for comm in hierarchical_communities[0]:
                layer_0_members.update(comm['members'])
            
            if nx_graph.number_of_nodes() > 0:
                metrics['layer_0_coverage'] = len(layer_0_members) / nx_graph.number_of_nodes()
        
        if hasattr(self.graph_store, 'entity_embeddings') and 0 in hierarchical_communities:
            coherence_scores = []
            
            for comm in hierarchical_communities[0]:
                members = [m for m in comm['members'] if m in self.graph_store.entity_embeddings]
                if len(members) > 1:
                    embeddings = [self.graph_store.entity_embeddings[m] for m in members]
                    
                    sims = []
                    for i in range(len(embeddings)):
                        for j in range(i + 1, len(embeddings)):
                            sim = cosine_similarity([embeddings[i]], [embeddings[j]])[0][0]
                            sims.append(sim)
                    
                    if sims:
                        coherence_scores.append(np.mean(sims))
            
            if coherence_scores:
                metrics['avg_community_coherence'] = np.mean(coherence_scores)
                metrics['std_community_coherence'] = np.std(coherence_scores)
        
        return metrics
    
    def _evaluate_chnsw_links(self):
        """
        NEW: Evaluate C-HNSW intra-layer and inter-layer link quality
        This is what was missing!
        """
        metrics = {}
        
        if not hasattr(self.graph_store, 'chnsw_index'):
            return metrics
        
        chnsw = self.graph_store.chnsw_index
        
        # Count total links
        total_intra_links = 0
        total_nodes = 0
        intra_link_weights = []
        
        # Evaluate each layer
        for layer_idx, layer_graph in enumerate(chnsw.layers):
            num_nodes = len(layer_graph.nodes())
            num_edges = len(layer_graph.edges())
            
            total_nodes += num_nodes
            total_intra_links += num_edges
            
            metrics[f'layer_{layer_idx}_nodes'] = num_nodes
            metrics[f'layer_{layer_idx}_intra_links'] = num_edges
            
            if num_nodes > 0:
                avg_degree = (2 * num_edges) / num_nodes
                metrics[f'layer_{layer_idx}_avg_degree'] = avg_degree
            
            # Analyze link weights (similarity)
            for u, v, data in layer_graph.edges(data=True):
                if 'similarity' in data:
                    intra_link_weights.append(data['similarity'])
        
        # Intra-layer metrics
        metrics['total_intra_layer_links'] = total_intra_links
        if intra_link_weights:
            metrics['avg_intra_link_similarity'] = np.mean(intra_link_weights)
            metrics['std_intra_link_similarity'] = np.std(intra_link_weights)
            metrics['min_intra_link_similarity'] = np.min(intra_link_weights)
            metrics['max_intra_link_similarity'] = np.max(intra_link_weights)
        
        # Inter-layer metrics
        inter_links = len(chnsw.inter_layer_links)
        metrics['total_inter_layer_links'] = inter_links
        
        if total_nodes > 0:
            metrics['avg_links_per_node'] = (total_intra_links + inter_links) / total_nodes
            metrics['intra_vs_inter_ratio'] = total_intra_links / (inter_links + 1e-10)
        
        # Hierarchical connectivity
        if len(chnsw.layers) > 1:
            expected_inter_links = sum(len(chnsw.layers[i].nodes()) 
                                      for i in range(1, len(chnsw.layers)))
            if expected_inter_links > 0:
                metrics['inter_layer_coverage'] = inter_links / expected_inter_links
        
        # Link quality
        if total_intra_links > 0 and total_nodes > 0:
            max_possible_links = sum(len(layer.nodes()) * (len(layer.nodes()) - 1) / 2 
                                    for layer in chnsw.layers)
            if max_possible_links > 0:
                metrics['link_density'] = total_intra_links / max_possible_links
        
        return metrics
    
    def _evaluate_information(self):
        """Evaluate information content"""
        metrics = {}
        
        nx_graph = self.graph_store._create_nx_graph()
        
        degrees = [d for n, d in nx_graph.degree()]
        degree_counts = defaultdict(int)
        for d in degrees:
            degree_counts[d] += 1
        
        total = len(degrees)
        if total > 0:
            degree_probs = [count / total for count in degree_counts.values()]
            metrics['degree_entropy'] = entropy(degree_probs)
        
        if hasattr(self.graph_store, 'hierarchical_communities') and 0 in self.graph_store.hierarchical_communities:
            community_sizes = [comm['size'] for comm in self.graph_store.hierarchical_communities[0]]
            total_members = sum(community_sizes)
            if total_members > 0:
                community_probs = [size / total_members for size in community_sizes]
                metrics['community_entropy'] = entropy(community_probs)
        
        return metrics
    
    def visualize_results(self, results_df):
        """Create visualization of evaluation results"""
        key_metrics = [
            'Structural_modularity',
            'Structural_avg_clustering_coefficient',
            'Semantic_semantic_separation_ratio',
            'Hierarchical_avg_community_coherence',
            'Hierarchical_compression_ratio',
            'CHNSW_avg_intra_link_similarity',
            'CHNSW_inter_layer_coverage'
        ]
        
        available_metrics = [m for m in key_metrics if m in results_df.apply(lambda x: f"{x['Category']}_{x['Metric']}", axis=1).values]
        
        if not available_metrics:
            print("No key metrics available for visualization")
            return
        
        plot_data = []
        for metric in available_metrics:
            row = results_df[results_df.apply(lambda x: f"{x['Category']}_{x['Metric']}", axis=1) == metric].iloc[0]
            plot_data.append({
                'Metric': row['Metric'].replace('_', ' ').title(),
                'Value': row['Value'],
                'Category': row['Category']
            })
        
        plot_df = pd.DataFrame(plot_data)
        
        # Create figure with subplots
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        
        # Bar chart
        colors = {'Structural': '#FFD700', 'Semantic': '#4169E1', 
                 'Hierarchical': '#DC143C', 'CHNSW': '#90EE90'}
        bar_colors = [colors.get(cat, '#888888') for cat in plot_df['Category']]
        
        axes[0].bar(range(len(plot_df)), plot_df['Value'], color=bar_colors)
        axes[0].set_xticks(range(len(plot_df)))
        axes[0].set_xticklabels(plot_df['Metric'], rotation=45, ha='right', fontsize=9)
        axes[0].set_ylabel('Score')
        axes[0].set_title('HA-GRAG Quality Metrics', fontsize=14, weight='bold')
        axes[0].grid(axis='y', alpha=0.3)
        
        for i, (bar, value) in enumerate(zip(axes[0].patches, plot_df['Value'])):
            height = bar.get_height()
            axes[0].text(bar.get_x() + bar.get_width()/2., height,
                        f'{value:.3f}', ha='center', va='bottom', fontsize=8)
        
        # Category breakdown
        category_counts = plot_df['Category'].value_counts()
        axes[1].pie(category_counts.values, labels=category_counts.index, 
                   autopct='%1.1f%%', colors=[colors.get(cat, '#888888') for cat in category_counts.index])
        axes[1].set_title('Metric Coverage by Category', fontsize=14, weight='bold')
        
        plt.tight_layout()
        plt.savefig('hagrag_enhanced_evaluation.png', dpi=300, bbox_inches='tight')
        plt.show()
        
        return fig

# ============================================================================
# RUN ENHANCED EVALUATION
# ============================================================================

# Initialize evaluator
evaluator = EnhancedGraphQualityEvaluator(graph_store)

# Run evaluation
results_df = evaluator.evaluate_all()

# Display results
print("\n" + "="*80)
print("GRAPH QUALITY METRICS (WITH HIERARCHICAL LINKS)")
print("="*80)
print(results_df.to_string(index=False))

# Save results
results_df.to_csv("hagrag_enhanced_evaluation.csv", index=False)
print(f"\n Results saved to: hagrag_enhanced_evaluation.csv")

# Visualize
print("\nGenerating visualization...")
fig = evaluator.visualize_results(results_df)

# Display enhanced summary
print("\n" + "="*80)
print("ENHANCED KEY FINDINGS (Including Hierarchical Links)")
print("="*80)

# Hierarchical structure
hierarchical_metrics = results_df[results_df['Category'] == 'Hierarchical']
print("\n1. HIERARCHICAL STRUCTURE:")
for _, row in hierarchical_metrics.iterrows():
    if row['Metric'] in ['num_layers', 'compression_ratio', 'avg_community_coherence']:
        if row['Metric'] == 'num_layers':
            interpretation = f" {int(row['Value'])} hierarchical layers built"
        elif row['Metric'] == 'compression_ratio':
            if row['Value'] < 0.3:
                interpretation = " Good hierarchical compression"
            else:
                interpretation = " Moderate hierarchical compression"
        elif row['Metric'] == 'avg_community_coherence':
            if row['Value'] > 0.7:
                interpretation = " Highly coherent communities"
            else:
                interpretation = " Moderately coherent communities"
        else:
            interpretation = ""
        
        print(f"   {row['Metric']}: {row['Value']:.3f} {interpretation}")

# C-HNSW links (NEW!)
chnsw_metrics = results_df[results_df['Category'] == 'CHNSW']
print("\n2. C-HNSW HIERARCHICAL LINKS:")
for _, row in chnsw_metrics.iterrows():
    if row['Metric'] in ['total_intra_layer_links', 'total_inter_layer_links', 
                          'avg_intra_link_similarity', 'inter_layer_coverage']:
        if row['Metric'] == 'total_intra_layer_links':
            interpretation = f" {int(row['Value'])} intra-layer links (green - within layers)"
        elif row['Metric'] == 'total_inter_layer_links':
            interpretation = f" {int(row['Value'])} inter-layer links (purple - between layers)"
        elif row['Metric'] == 'avg_intra_link_similarity':
            if row['Value'] > 0.5:
                interpretation = " High intra-layer similarity"
            else:
                interpretation = " Moderate intra-layer similarity"
        elif row['Metric'] == 'inter_layer_coverage':
            if row['Value'] > 0.8:
                interpretation = " Excellent inter-layer connectivity"
            else:
                interpretation = " Good inter-layer connectivity"
        else:
            interpretation = ""
        
        print(f"   {row['Metric']}: {row['Value']:.3f} {interpretation}")

print("\n" + "="*80)
print(" EVALUATION COMPLETE WITH HIERARCHICAL LINK ANALYSIS!")
print("="*80)

In [ ]:
# First remove any old versions
!pip uninstall -y llama_index gpt_index

# Install the latest full package (with integrations like Ollama, HuggingFace, etc.)
!pip install -U llama-index
!pip install llama-index-llms-huggingface llama-index-llms-ollama
!pip install rouge-score


In [ ]:
# Install LangChain and the community LLM integrations
!pip install -U langchain langchain_community


In [ ]:
from llama_index.core.evaluation import (
    FaithfulnessEvaluator,
    RelevancyEvaluator,
    CorrectnessEvaluator
)

# These imports will now work
from llama_index.llms.ollama import Ollama
from llama_index.llms.huggingface import HuggingFaceLLM
from rouge_score import rouge_scorer



In [ ]:
# Install specific LlamaIndex LLM packages
!pip install llama-index-llms-ollama llama-index-llms-huggingface

# Or install all LlamaIndex LLM integrations
!pip install 'llama-index[llms]'

In [ ]:
!pip install -U llama-index


In [ ]:
!pip install sacremoses
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL_NAME = "microsoft/BioGPT"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
print("Model loaded successfully")

In [ ]:
!pip show transformers

In [24]:
import os
import json
import pandas as pd
import logging
import matplotlib.pyplot as plt
from tqdm import tqdm
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from typing import List, Dict, Any
import nltk
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from langchain_core.output_parsers import StrOutputParser
from neo4j import GraphDatabase
from transformers import AutoTokenizer, AutoModelForCausalLM,  BitsAndBytesConfig
import torch
from transformers import pipeline as hf_pipeline
# Ensure NLTK punkt is downloaded
try:
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)
except Exception as e:
    logging.error(f"Failed to download NLTK punkt: {e}")
    raise

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Set Hugging Face cache directory
os.environ["HF_HOME"] = "./workspace/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "./workspace/hf_cache"

def check_neo4j_graph(neo4j_uri: str, neo4j_username: str, neo4j_password: str) -> Dict[str, Any]:
    """Check the Neo4j database for nodes, relationships, and diabetes-related entities."""
    try:
        driver = GraphDatabase.driver(neo4j_uri, auth=(neo4j_username, neo4j_password))
        with driver.session() as session:
            entity_result = session.run("MATCH (e:Entity) RETURN count(e) AS entity_count")
            entity_count = entity_result.single()["entity_count"]
            rel_result = session.run("MATCH ()-[r:RELATION]->() RETURN count(r) AS rel_count")
            rel_count = rel_result.single()["rel_count"]
            sample_entities = session.run(
                "MATCH (e:Entity) RETURN e.name, e.type, e.description LIMIT 5"
            ).data()
            sample_rels = session.run(
                "MATCH (e1)-[r:RELATION]->(e2) RETURN e1.name, r.type, e2.name LIMIT 5"
            ).data()
            diabetes_entities = session.run(
                "MATCH (e:Entity) WHERE toLower(e.name) CONTAINS 'diabetes' OR toLower(e.description) CONTAINS 'diabetes' "
                "RETURN e.name, e.type, e.description LIMIT 10"
            ).data()
            family_history_entities = session.run(
                "MATCH (e:Entity) WHERE toLower(e.name) CONTAINS 'family history' OR toLower(e.description) CONTAINS 'family history' "
                "RETURN e.name, e.type, e.description LIMIT 10"
            ).data()
        driver.close()
        logging.info(f"Neo4j Graph Stats: {entity_count} entities, {rel_count} relationships")
        logging.info(f"Sample Entities: {sample_entities}")
        logging.info(f"Sample Relationships: {sample_rels}")
        logging.info(f"Diabetes-Related Entities: {diabetes_entities}")
        logging.info(f"Family History Entities: {family_history_entities}")
        return {
            "entity_count": entity_count,
            "rel_count": rel_count,
            "sample_entities": sample_entities,
            "sample_relationships": sample_rels,
            "diabetes_entities": diabetes_entities,
            "family_history_entities": family_history_entities
        }
    except Exception as e:
        logging.error(f"Neo4j connection error: {e}")
        return {
            "entity_count": 0,
            "rel_count": 0,
            "sample_entities": [],
            "sample_relationships": [],
            "diabetes_entities": [],
            "family_history_entities": []
        }

def sample_dataset(dataset_path: str, num_samples: int = 3) -> List[Dict]:
    """Sample entries from the evaluation dataset."""
    try:
        with open(dataset_path, 'r') as f:
            data = [json.loads(line) for line in f]
        samples = data[:min(num_samples, len(data))]
        for i, sample in enumerate(samples):
            logging.info(f"Dataset Sample {i+1}: {sample}")
        return samples
    except Exception as e:
        logging.error(f"Error sampling dataset: {e}")
        return []

def test_query_engine(query_engine, dataset_path: str) -> Dict[str, Any]:
    """Test the query engine with questions from the dataset."""
    samples = sample_dataset(dataset_path, num_samples=2)
    test_queries = [sample["question"] for sample in samples]
    results = {}
    for test_query in test_queries:
        try:
            result = query_engine.query(test_query)
            logging.info(f"Test Query: {test_query}")
            logging.info(f"Query Engine Output Type: {type(result)}")
            logging.info(f"Query Engine Output: {result}")
            if isinstance(result, dict):
                response_text = result.get('response', 'No response')
                hierarchical_info = result.get('hierarchical_info', {})
                logging.info(f"Response: {response_text}")
                logging.info(f"Hierarchical Info: {hierarchical_info}")
            else:
                response_text = str(result)
                logging.info(f"Response: {response_text}")
                logging.info("Hierarchical Info: Not available (string response)")
            results[test_query] = result
        except Exception as e:
            logging.error(f"Error testing query '{test_query}': {e}")
            results[test_query] = {}
    return results

def load_eval_dataset(json_file_path: str, max_queries: int = 5) -> List[Dict]:
    """Load evaluation dataset from JSONL file, limited to max_queries."""
    try:
        with open(json_file_path, 'r') as f:
            data = [json.loads(line) for line in f]
        data = data[:min(max_queries, len(data))]
        logging.info(f"Loaded {len(data)} evaluation samples (limited to {max_queries})")
        return data
    except Exception as e:
        logging.error(f"Error loading dataset: {e}")
        raise

def init_evaluators(pipeline, model_name: str = "mistralai/Mixtral-7B-Instruct-v0.1") -> tuple:
    """Initialize evaluators, reusing pipeline's LLM if available, else load a smaller model."""
    try:
        # Try to reuse pipeline's LLM
        if hasattr(pipeline, 'llm') and pipeline.llm is not None:
            logging.info("Reusing LLM from pipeline for evaluation...")
            llm = pipeline.llm
        else:
            logging.info(f"Pipeline LLM not found, initializing {model_name} with 4-bit quantization...")
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True
            )
            tokenizer = AutoTokenizer.from_pretrained(model_name, token=os.getenv("HF_TOKEN"))
            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                token=os.getenv("HF_TOKEN"),
                quantization_config=quantization_config,
                device_map="auto",
                low_cpu_mem_usage=True
            )
            



            text_generator = hf_pipeline(
                "text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=128,
                temperature=0.1,
                do_sample=True,
                top_p=0.95,
                repetition_penalty=1.2,
                return_full_text=False
            )
            llm = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=text_generator))

        class CustomFaithfulnessEvaluator:
            def __init__(self, llm):
                self.llm = llm
                self.prompt = ChatPromptTemplate.from_messages([
                    ("system", "Determine if the response is faithful to the provided contexts. "
                               "Return a JSON object with 'passing' (boolean) and 'feedback' (string)."),
                    ("human", "Query: {query}\nResponse: {response}\nContexts: {contexts}")
                ])
                self.chain = self.prompt | self.llm | StrOutputParser()

            def evaluate(self, query, response, contexts):
                try:
                    result = json.loads(self.chain.invoke({
                        "query": query,
                        "response": response,
                        "contexts": "\n".join(contexts)
                    }))
                    return type('Result', (), {
                        'passing': result.get('passing', False),
                        'feedback': result.get('feedback', 'No feedback')
                    })()
                except Exception as e:
                    logging.warning(f"Faithfulness evaluation failed: {e}")
                    return type('Result', (), {'passing': False, 'feedback': f"Error: {e}"})()

        class CustomRelevancyEvaluator:
            def __init__(self, llm):
                self.llm = llm
                self.prompt = ChatPromptTemplate.from_messages([
                    ("system", "Determine if the response is relevant to the query based on the contexts. "
                               "Return a JSON object with 'passing' (boolean) and 'feedback' (string)."),
                    ("human", "Query: {query}\nResponse: {response}\nContexts: {contexts}")
                ])
                self.chain = self.prompt | self.llm | StrOutputParser()

            def evaluate(self, query, response, contexts):
                try:
                    result = json.loads(self.chain.invoke({
                        "query": query,
                        "response": response,
                        "contexts": "\n".join(contexts)
                    }))
                    return type('Result', (), {
                        'passing': result.get('passing', False),
                        'feedback': result.get('feedback', 'No feedback')
                    })()
                except Exception as e:
                    logging.warning(f"Relevancy evaluation failed: {e}")
                    return type('Result', (), {'passing': False, 'feedback': f"Error: {e}"})()

        class CustomCorrectnessEvaluator:
            def __init__(self, llm):
                self.llm = llm
                self.prompt = ChatPromptTemplate.from_messages([
                    ("system", "Evaluate the correctness of the response compared to the reference answer. "
                               "Return a JSON object with 'score' (0-5), 'passing' (boolean, true if score >= 4), "
                               "and 'feedback' (string)."),
                    ("human", "Query: {query}\nResponse: {response}\nReference: {reference}")
                ])
                self.chain = self.prompt | self.llm | StrOutputParser()

            def evaluate(self, query, response, reference):
                try:
                    result = json.loads(self.chain.invoke({
                        "query": query,
                        "response": response,
                        "reference": reference
                    }))
                    score = result.get('score', 0.0)
                    return type('Result', (), {
                        'score': score,
                        'passing': score >= 4,
                        'feedback': result.get('feedback', 'No feedback')
                    })()
                except Exception as e:
                    logging.warning(f"Correctness evaluation failed: {e}")
                    return type('Result', (), {
                        'score': 0.0,
                        'passing': False,
                        'feedback': f"Error: {e}"
                    })()

        faithfulness_evaluator = CustomFaithfulnessEvaluator(llm)
        relevancy_evaluator = CustomRelevancyEvaluator(llm)
        correctness_evaluator = CustomCorrectnessEvaluator(llm)
        logging.info("Successfully initialized evaluators")
        return faithfulness_evaluator, relevancy_evaluator, correctness_evaluator, llm
    except Exception as e:
        logging.error(f"Error initializing evaluators: {e}")
        raise

def compute_rouge_bleu(response: str, reference: str) -> Dict[str, float]:
    """Compute ROUGE and BLEU scores."""
    try:
        scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
        rouge_scores = scorer.score(reference, response)
        rouge_l = rouge_scores['rougeL'].fmeasure
        reference_tokens = [nltk.word_tokenize(reference.lower())]
        response_tokens = nltk.word_tokenize(response.lower())
        bleu_score = sentence_bleu(reference_tokens, response_tokens, weights=(0.25, 0.25, 0.25, 0.25),
                                  smoothing_function=SmoothingFunction().method1)
        return {"rouge_l": rouge_l, "bleu_4": bleu_score}
    except Exception as e:
        logging.warning(f"Error computing ROUGE/BLEU: {e}")
        return {"rouge_l": 0.0, "bleu_4": 0.0}

def evaluate_node_relevance(query: str, hierarchical_info: Dict[int, List[Dict]], llm) -> float:
    """Evaluate relevance of retrieved graph nodes to the query."""
    try:
        relevant_nodes = 0
        total_nodes = 0
        prompt = ChatPromptTemplate.from_messages([
            ("system", "Determine if the provided node content is relevant to the query. "
                       "Return 'yes' or 'no'."),
            ("human", "Query: {query}\nNode Content: {content}")
        ])
        chain = prompt | llm | StrOutputParser()
        for layer, nodes in hierarchical_info.items():
            for node in nodes:
                total_nodes += 1
                content = node.get('content', {}).get('summary', node.get('content', {}).get('description', ''))
                if not content:
                    continue
                result = chain.invoke({"query": query, "content": content}).strip().lower()
                if result == "yes":
                    relevant_nodes += 1
        return relevant_nodes / total_nodes if total_nodes > 0 else 0.0
    except Exception as e:
        logging.warning(f"Error evaluating node relevance: {e}")
        return 0.0

def evaluate_query(query_engine, query_data: Dict, evaluators: tuple, llm) -> Dict[str, Any]:
    """Evaluate a single query against the ArchRAG query engine."""
    query = query_data["question"]
    reference = query_data["answer"]
    context = query_data.get("context", "")
    try:
        result = query_engine.query(query)
        if isinstance(result, dict):
            response_text = result.get("response", "")
            hierarchical_info = result.get("hierarchical_info", {})
        else:
            response_text = str(result)
            hierarchical_info = {}
        if not response_text or response_text.startswith("I couldn't find relevant information"):
            logging.warning(f"Empty or invalid response for query: {query}")
            response_text = ""
            logging.info(f"Query Result: {result}")
            logging.info(f"Retrieved Nodes: {hierarchical_info}")
        faithfulness_evaluator, relevancy_evaluator, correctness_evaluator = evaluators
        eval_contexts = []
        if hierarchical_info:
            for layer, nodes in hierarchical_info.items():
                for node in nodes:
                    content = node.get('content', {})
                    content = content.get('summary', content.get('description', '')) if isinstance(content, dict) else str(content)
                    if content:
                        eval_contexts.append(content)
        if not eval_contexts:
            eval_contexts = [context] if context else ["No context available"]
        faithfulness_result = faithfulness_evaluator.evaluate(
            query=query,
            response=response_text,
            contexts=eval_contexts
        )
        relevancy_result = relevancy_evaluator.evaluate(
            query=query,
            response=response_text,
            contexts=eval_contexts
        )
        source_relevancy_results = []
        if hierarchical_info:
            for layer, nodes in hierarchical_info.items():
                for node in nodes:
                    content = node.get('content', {})
                    content = content.get('summary', content.get('description', '')) if isinstance(content, dict) else str(content)
                    if not content:
                        continue
                    try:
                        node_result = relevancy_evaluator.evaluate(
                            query=query,
                            response=response_text,
                            contexts=[content]
                        )
                        source_relevancy_results.append(node_result.passing)
                    except Exception as e:
                        logging.warning(f"Error evaluating node relevancy: {e}")
                        continue
        source_relevancy_score = (sum(source_relevancy_results) / len(source_relevancy_results)
                                 if source_relevancy_results else 0.0)
        correctness_result = correctness_evaluator.evaluate(
            query=query,
            response=response_text,
            reference=reference
        )
        rouge_bleu_scores = compute_rouge_bleu(response_text, reference)
        node_precision = evaluate_node_relevance(query, hierarchical_info, llm) if hierarchical_info else 0.0
        community_coverage = 0.0
        if hierarchical_info:
            valid_layers = [l for l, nodes in hierarchical_info.items() if nodes]
            community_coverage = len(valid_layers) / (len(hierarchical_info) or 1)
        return {
            "query": query,
            "response": response_text,
            "reference": reference,
            "faithfulness_score": 1 if faithfulness_result.passing else 0,
            "faithfulness_feedback": faithfulness_result.feedback,
            "relevancy_score": 1 if relevancy_result.passing else 0,
            "relevancy_feedback": relevancy_result.feedback,
            "source_relevancy_score": source_relevancy_score,
            "correctness_score": correctness_result.score,
            "correctness_passing": 1 if correctness_result.passing else 0,
            "correctness_feedback": correctness_result.feedback,
            "rouge_l": rouge_bleu_scores["rouge_l"],
            "bleu_4": rouge_bleu_scores["bleu_4"],
            "node_precision": node_precision,
            "community_coverage": community_coverage
        }
    except Exception as e:
        logging.error(f"Error evaluating query '{query}': {e}")
        import traceback
        logging.error(f"Traceback: {traceback.format_exc()}")
        return {
            "query": query,
            "response": "",
            "reference": reference,
            "faithfulness_score": 0,
            "faithfulness_feedback": f"Evaluation error: {e}",
            "relevancy_score": 0,
            "relevancy_feedback": f"Evaluation error: {e}",
            "source_relevancy_score": 0.0,
            "correctness_score": 0.0,
            "correctness_passing": 0,
            "correctness_feedback": f"Evaluation error: {e}",
            "rouge_l": 0.0,
            "bleu_4": 0.0,
            "node_precision": 0.0,
            "community_coverage": 0.0
        }

def create_visualizations(results_df: pd.DataFrame, metrics: Dict[str, float]) -> None:
    """Create visualizations for evaluation metrics."""
    try:
        valid_metrics = {k: v for k, v in metrics.items() if not pd.isna(v)}
        metric_names = list(valid_metrics.keys())
        metric_values = list(valid_metrics.values())
        plt.figure(figsize=(12, 6))
        bars = plt.bar(metric_names, metric_values)
        plt.ylim(0, 1.1)
        plt.title('ArchRAG Evaluation Metrics')
        plt.ylabel('Score')
        plt.xticks(rotation=45)
        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                     f'{height:.2f}', ha='center', va='bottom')
        plt.tight_layout()
        plt.savefig('./workspace/test/archrag_evaluation_metrics_test.png')
        plt.close()
        if 'correctness_score' in results_df.columns:
            plt.figure(figsize=(8, 6))
            plt.hist(results_df['correctness_score'].dropna(), bins=10, alpha=0.7)
            plt.title('Distribution of Correctness Scores')
            plt.xlabel('Correctness Score')
            plt.ylabel('Count')
            plt.savefig('./workspace/test/correctness_distribution_test.png')
            plt.close()
        if 'node_precision' in results_df.columns:
            plt.figure(figsize=(8, 6))
            plt.hist(results_df['node_precision'].dropna(), bins=10, alpha=0.7)
            plt.title('Distribution of Node Retrieval Precision')
            plt.xlabel('Node Precision')
            plt.ylabel('Count')
            plt.savefig('./workspace/test/node_precision_distribution_test.png')
            plt.close()
        if 'rouge_l' in results_df.columns and 'node_precision' in results_df.columns:
            plt.figure(figsize=(8, 6))
            plt.scatter(results_df['rouge_l'], results_df['node_precision'], alpha=0.5)
            plt.title('ROUGE-L vs Node Retrieval Precision')
            plt.xlabel('ROUGE-L Score')
            plt.ylabel('Node Precision')
            plt.savefig('./workspace/test/rouge_vs_node_test.png')
            plt.close()
    except Exception as e:
        logging.error(f"Error creating visualizations: {e}")

def evaluate_archrag_system(pipeline, dataset_path: str, config: Dict[str, str]) -> tuple:
    """Evaluate the ArchRAG system using the existing knowledge graph and dataset."""
    logging.info("Starting ArchRAG evaluation...")
    pdf_texts = []  # Skip PDF reading
    dataset_samples = sample_dataset(dataset_path)
    graph_stats = check_neo4j_graph(config["neo4j_uri"], config["neo4j_username"], config["neo4j_password"])
    query_engine = pipeline.query_engine
    test_query_engine(query_engine, dataset_path)
    eval_dataset = load_eval_dataset(dataset_path, max_queries=5)
    evaluators = init_evaluators(pipeline)
    llm = evaluators[3]
    results = []
    for query_data in tqdm(eval_dataset, desc="Evaluating queries"):
        result = evaluate_query(query_engine, query_data, evaluators[:3], llm)
        results.append(result)
    results_df = pd.DataFrame(results)
    metrics = {
        "Faithfulness": results_df["faithfulness_score"].mean(),
        "Relevancy": results_df["relevancy_score"].mean(),
        "Source Relevancy": results_df["source_relevancy_score"].mean() if not results_df["source_relevancy_score"].empty else 0.0,
        "Correctness Score": results_df["correctness_score"].mean(),
        "Correctness Pass Rate": results_df["correctness_passing"].mean(),
        "ROUGE-L": results_df["rouge_l"].mean(),
        "BLEU-4": results_df["bleu_4"].mean(),
        "Node Precision": results_df["node_precision"].mean(),
        "Community Coverage": results_df["community_coverage"].mean()
    }
    metrics_df = pd.DataFrame({
        "Metric": list(metrics.keys()),
        "Score": list(metrics.values())
    })
    create_visualizations(results_df, metrics)
    results_df.to_csv("./workspace/test/archrag_evaluation_detailed_results_test.csv", index=False)
    metrics_df.to_csv("./workspace/test/archrag_evaluation_metrics_test.csv", index=False)
    logging.info("\nArchRAG Evaluation Summary:")
    logging.info(metrics_df.to_string(index=False))
    logging.info("\nDetailed results saved to './workspace/test/archrag_evaluation_detailed_results_test.csv'")
    logging.info("Metrics summary saved to './workspace/test/archrag_evaluation_metrics_test.csv'")
    logging.info("Visualizations saved in './workspace/test/'")
    return results_df, metrics_df, graph_stats, pdf_texts

if __name__ == "__main__":
    config = {
        "pdf_dir": "/home/871323/RAG/test/150papers/150papers",
        "neo4j_uri": os.getenv("NEO4J_URI"),
        "neo4j_username": os.getenv("NEO4J_USERNAME", "neo4j"),
        "neo4j_password": os.getenv("NEO4J_PASSWORD"),
    }
    dataset_path = "/home/871323/RAG/test/100diabetes_qa_dataset.json"
    pipeline = ArchRAGPipeline(**config)
    # Ensure 'pipeline' is your initialized ArchRAGPipeline object
    
    try:
        results_df, metrics_df, graph_stats, pdf_texts = evaluate_archrag_system(
            pipeline=pipeline,
            dataset_path=dataset_path,
            config=config
        )
    except Exception as e:
        logging.error(f"Evaluation failed: {e}")
        import traceback
        logging.error(f"Traceback: {traceback.format_exc()}")

INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cuda:0
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: all-MiniLM-L6-v2
/home/871323/.local/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Error loading quantized model: Failed to import transformers.integrations.bitsandbytes because of the following error (look up to see its traceback):
cannot import name 'tarfile' from 'backports' (/home/ViperAppsFiles/python/anaconda/miniconda/3.10/lib/python3.10/site-packages/backports/__init__.py)
Falling back to non-quantized model...


INFO:accelerate.utils.modeling:We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

INFO:root:Starting ArchRAG evaluation...
INFO:root:Dataset Sample 1: {'question': 'Prompting Primary Care Providers about Increased Patient Risk As a Result of Family History: Does It Work?', 'answer': "No change occurred upon instituting simple, at-the-visit family history prompts geared to improve PCPs' ability to identify patients at high risk for 6 common conditions. The results are both surprising and disappointing. Further studies should examine physicians' perception of the utility of prompts for family history risk.", 'context': 'Electronic health records have the potential to facilitate family history use by primary care physicians (PCPs) to provide personalized care. The objective of this study was to determine whether automated, at-the-visit tailored prompts about family history risk change PCP behavior. Automated, tailored prompts highlighting familial risk for heart disease, stroke, diabetes, and breast, colorectal, or ovarian cancer were implemented during 2011 to 2012. M

INFO:root:Family History Entities: []
INFO:root:Dataset Sample 1: {'question': 'Prompting Primary Care Providers about Increased Patient Risk As a Result of Family History: Does It Work?', 'answer': "No change occurred upon instituting simple, at-the-visit family history prompts geared to improve PCPs' ability to identify patients at high risk for 6 common conditions. The results are both surprising and disappointing. Further studies should examine physicians' perception of the utility of prompts for family history risk.", 'context': 'Electronic health records have the potential to facilitate family history use by primary care physicians (PCPs) to provide personalized care. The objective of this study was to determine whether automated, at-the-visit tailored prompts about family history risk change PCP behavior. Automated, tailored prompts highlighting familial risk for heart disease, stroke, diabetes, and breast, colorectal, or ovarian cancer were implemented during 2011 to 2012. Medi

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:Test Query: Prompting Primary Care Providers about Increased Patient Risk As a Result of Family History: Does It Work?
INFO:root:Query Engine Output Type: <class 'dict'>
INFO:root:Query Engine Output: {'query': 'Prompting Primary Care Providers about Increased Patient Risk As a Result of Family History: Does It Work?', 'response': "I couldn't find relevant information to answer your query.", 'hierarchical_info': {}, 'filtered_info': [], 'layers_searched': []}
INFO:root:Response: I couldn't find relevant information to answer your query.
INFO:root:Hierarchical Info: {}
INFO:root:Processing query: Do general practice characteristics influence uptake of an information technology (IT) innovation in primary care?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:root:Test Query: Do general practice characteristics influence uptake of an information technology (IT) innovation in primary care?
INFO:root:Query Engine Output Type: <class 'dict'>
INFO:root:Query Engine Output: {'query': 'Do general practice characteristics influence uptake of an information technology (IT) innovation in primary care?', 'response': "I couldn't find relevant information to answer your query.", 'hierarchical_info': {}, 'filtered_info': [], 'layers_searched': []}
INFO:root:Response: I couldn't find relevant information to answer your query.
INFO:root:Hierarchical Info: {}
INFO:root:Loaded 5 evaluation samples (limited to 5)
INFO:root:Pipeline LLM not found, initializing mistralai/Mixtral-7B-Instruct-v0.1 with 4-bit quantization...
ERROR:root:Error initializing evaluators: mistralai/Mixtral-7B-Instruct-v0.1 is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token h

In [25]:
import json
import pandas as pd
import logging
import matplotlib.pyplot as plt
from tqdm import tqdm
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from typing import List, Dict, Any
import nltk
import os
from neo4j import GraphDatabase
from PyPDF2 import PdfReader
import torch
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, 
    AutoModelForSequenceClassification, pipeline
)
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import re

# Ensure NLTK punkt is downloaded
try:
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)
except Exception as e:
    logging.error(f"Failed to download NLTK punkt: {e}")
    raise

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

class OpenSourceEvaluator:
    """Open source evaluator using Hugging Face models."""
    
    def __init__(self, model_name="microsoft/DialoGPT-medium", device="cuda" if torch.cuda.is_available() else "cpu"):
        self.device = device
        logging.info(f"Initializing evaluators on device: {self.device}")
        
        # Initialize models
        self._init_models(model_name)
        
    def _init_models(self, model_name):
        """Initialize all required models."""
        try:
            # For general text generation and evaluation
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token
                
            self.model = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=torch.float16 if self.device == "cuda" else torch.float32,
                device_map="auto" if self.device == "cuda" else None
            )
            
            # Sentence transformer for semantic similarity
            self.sentence_model = SentenceTransformer('all-MiniLM-L6-v2')
            
            # NLI model for faithfulness evaluation
            self.nli_model = pipeline(
                "text-classification",
                model="microsoft/DialoGPT-medium",
                device=0 if self.device == "cuda" else -1,
                return_all_scores=True
            )
            
            logging.info("All models initialized successfully")
            
        except Exception as e:
            logging.error(f"Error initializing models: {e}")
            # Fallback to CPU models
            self.device = "cpu"
            self._init_fallback_models()
    
    def _init_fallback_models(self):
        """Initialize fallback models for CPU."""
        try:
            self.tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token
                
            self.model = AutoModelForCausalLM.from_pretrained("distilgpt2")
            self.sentence_model = SentenceTransformer('all-MiniLM-L6-v2')
            
            # Simple NLI pipeline
            self.nli_model = pipeline(
                "zero-shot-classification",
                model="facebook/bart-large-mnli",
                device=-1
            )
            
            logging.info("Fallback models initialized successfully")
            
        except Exception as e:
            logging.error(f"Error initializing fallback models: {e}")
            raise

class OpenSourceFaithfulnessEvaluator:
    """Open source faithfulness evaluator."""
    
    def __init__(self, evaluator: OpenSourceEvaluator):
        self.evaluator = evaluator
        
    def evaluate(self, query: str, response: str, contexts: List[str]):
        """Evaluate faithfulness using semantic similarity."""
        try:
            if not response or not contexts:
                return type('Result', (), {'passing': False, 'feedback': 'Empty response or contexts'})()
            
            # Encode response and contexts
            response_embedding = self.evaluator.sentence_model.encode([response])
            context_embeddings = self.evaluator.sentence_model.encode(contexts)
            
            # Calculate similarity scores
            similarities = cosine_similarity(response_embedding, context_embeddings)[0]
            max_similarity = np.max(similarities)
            
            # Threshold for faithfulness (can be adjusted)
            threshold = 0.3
            passing = max_similarity > threshold
            
            feedback = f"Max similarity: {max_similarity:.3f}, Threshold: {threshold}"
            
            return type('Result', (), {'passing': passing, 'feedback': feedback})()
            
        except Exception as e:
            logging.error(f"Error in faithfulness evaluation: {e}")
            return type('Result', (), {'passing': False, 'feedback': f'Error: {e}'})()

class OpenSourceRelevancyEvaluator:
    """Open source relevancy evaluator."""
    
    def __init__(self, evaluator: OpenSourceEvaluator):
        self.evaluator = evaluator
        
    def evaluate(self, query: str, response: str, contexts: List[str]):
        """Evaluate relevancy using semantic similarity."""
        try:
            if not response or not query:
                return type('Result', (), {'passing': False, 'feedback': 'Empty response or query'})()
            
            # Encode query and response
            query_embedding = self.evaluator.sentence_model.encode([query])
            response_embedding = self.evaluator.sentence_model.encode([response])
            
            # Calculate similarity
            similarity = cosine_similarity(query_embedding, response_embedding)[0][0]
            
            # Threshold for relevancy
            threshold = 0.4
            passing = similarity > threshold
            
            feedback = f"Query-response similarity: {similarity:.3f}, Threshold: {threshold}"
            
            return type('Result', (), {'passing': passing, 'feedback': feedback})()
            
        except Exception as e:
            logging.error(f"Error in relevancy evaluation: {e}")
            return type('Result', (), {'passing': False, 'feedback': f'Error: {e}'})()

class OpenSourceCorrectnessEvaluator:
    """Open source correctness evaluator."""
    
    def __init__(self, evaluator: OpenSourceEvaluator):
        self.evaluator = evaluator
        
    def evaluate(self, query: str, response: str, reference: str):
        """Evaluate correctness using semantic similarity and text overlap."""
        try:
            if not response or not reference:
                return type('Result', (), {
                    'passing': False, 
                    'score': 0.0, 
                    'feedback': 'Empty response or reference'
                })()
            
            # Semantic similarity
            response_embedding = self.evaluator.sentence_model.encode([response])
            reference_embedding = self.evaluator.sentence_model.encode([reference])
            semantic_sim = cosine_similarity(response_embedding, reference_embedding)[0][0]
            
            # ROUGE-L score for text overlap
            scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
            rouge_scores = scorer.score(reference, response)
            rouge_l = rouge_scores['rougeL'].fmeasure
            
            # Combined score
            combined_score = (semantic_sim + rouge_l) / 2
            
            # Threshold for correctness
            threshold = 0.3
            passing = combined_score > threshold
            
            feedback = f"Semantic sim: {semantic_sim:.3f}, ROUGE-L: {rouge_l:.3f}, Combined: {combined_score:.3f}"
            
            return type('Result', (), {
                'passing': passing, 
                'score': combined_score, 
                'feedback': feedback
            })()
            
        except Exception as e:
            logging.error(f"Error in correctness evaluation: {e}")
            return type('Result', (), {
                'passing': False, 
                'score': 0.0, 
                'feedback': f'Error: {e}'
            })()

def check_neo4j_graph(neo4j_uri: str, neo4j_username: str, neo4j_password: str) -> Dict[str, Any]:
    """Check the Neo4j database for nodes, relationships, and sample data."""
    try:
        driver = GraphDatabase.driver(neo4j_uri, auth=(neo4j_username, neo4j_password))
        with driver.session() as session:
            # Count entities
            entity_result = session.run("MATCH (e:Entity) RETURN count(e) AS entity_count")
            entity_count = entity_result.single()["entity_count"]
            # Count relationships
            rel_result = session.run("MATCH ()-[r:RELATION]->() RETURN count(r) AS rel_count")
            rel_count = rel_result.single()["rel_count"]
            # Sample entities
            sample_entities = session.run(
                "MATCH (e:Entity) RETURN e.name, e.type, e.description LIMIT 5"
            ).data()
            # Sample relationships
            sample_rels = session.run(
                "MATCH (e1)-[r:RELATION]->(e2) RETURN e1.name, r.type, e2.name LIMIT 5"
            ).data()
        driver.close()
        logging.info(f"Neo4j Graph Stats: {entity_count} entities, {rel_count} relationships")
        logging.info(f"Sample Entities: {sample_entities}")
        logging.info(f"Sample Relationships: {sample_rels}")
        return {
            "entity_count": entity_count,
            "rel_count": rel_count,
            "sample_entities": sample_entities,
            "sample_relationships": sample_rels
        }
    except Exception as e:
        logging.error(f"Neo4j connection error: {e}")
        return {
            "entity_count": 0,
            "rel_count": 0,
            "sample_entities": [],
            "sample_relationships": []
        }

def log_pdf_content(pdf_dir: str) -> List[str]:
    """Log the content of PDFs in the directory."""
    try:
        pdf_texts = []
        for pdf_file in os.listdir(pdf_dir):
            if pdf_file.endswith(".pdf"):
                pdf_path = os.path.join(pdf_dir, pdf_file)
                logging.info(f"Extracting text from PDF: {pdf_file}")
                try:
                    reader = PdfReader(pdf_path)
                    text = ""
                    for page in reader.pages:
                        page_text = page.extract_text() or ""
                        text += page_text
                    pdf_texts.append(text[:1000])  # Log first 1000 chars
                    logging.info(f"Sample text from {pdf_file}: {text[:500]}...")
                except Exception as e:
                    logging.warning(f"Error extracting text from {pdf_file}: {e}")
        return pdf_texts
    except Exception as e:
        logging.error(f"Error accessing PDF directory: {e}")
        return []

def sample_dataset(dataset_path: str, num_samples: int = 3) -> List[Dict]:
    """Sample entries from the evaluation dataset."""
    try:
        with open(dataset_path, 'r') as f:
            data = [json.loads(line) for line in f]
        samples = data[:min(num_samples, len(data))]
        for i, sample in enumerate(samples):
            logging.info(f"Dataset Sample {i+1}: {sample}")
        return samples
    except Exception as e:
        logging.error(f"Error sampling dataset: {e}")
        return []

def test_query_engine(query_engine, test_query: str = "What is type 1 diabetes?"):
    """Test the query engine with a sample query."""
    try:
        result = query_engine.query(test_query)
        logging.info(f"Test Query: {test_query}")
        logging.info(f"Query Engine Output Type: {type(result)}")
        logging.info(f"Query Engine Output: {result}")

        # Handle both dict and string responses
        if isinstance(result, dict):
            response_text = result.get('response', 'No response')
            hierarchical_info = result.get('hierarchical_info', {})
            logging.info(f"Response: {response_text}")
            logging.info(f"Hierarchical Info: {hierarchical_info}")
        else:
            # Result is a string
            response_text = str(result)
            logging.info(f"Response: {response_text}")
            logging.info("Hierarchical Info: Not available (string response)")

        return result
    except Exception as e:
        logging.error(f"Error testing query engine: {e}")
        return {}

def load_eval_dataset(json_file_path: str) -> List[Dict]:
    """Load evaluation dataset from JSONL file."""
    try:
        with open(json_file_path, 'r') as f:
            data = [json.loads(line) for line in f]
        logging.info(f"Loaded {len(data)} evaluation samples")
        return data
    except Exception as e:
        logging.error(f"Error loading dataset: {e}")
        raise

def init_evaluators(model_name: str = "microsoft/DialoGPT-medium"):
    """Initialize open-source evaluators."""
    try:
        base_evaluator = OpenSourceEvaluator(model_name)
        faithfulness_evaluator = OpenSourceFaithfulnessEvaluator(base_evaluator)
        relevancy_evaluator = OpenSourceRelevancyEvaluator(base_evaluator)
        correctness_evaluator = OpenSourceCorrectnessEvaluator(base_evaluator)
        return faithfulness_evaluator, relevancy_evaluator, correctness_evaluator
    except Exception as e:
        logging.error(f"Error initializing evaluators: {e}")
        raise

def compute_rouge_bleu(response: str, reference: str) -> Dict[str, float]:
    """Compute ROUGE and BLEU scores."""
    try:
        scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
        rouge_scores = scorer.score(reference, response)
        rouge_l = rouge_scores['rougeL'].fmeasure

        reference_tokens = [nltk.word_tokenize(reference.lower())]
        response_tokens = nltk.word_tokenize(response.lower())
        bleu_score = sentence_bleu(reference_tokens, response_tokens, weights=(0.25, 0.25, 0.25, 0.25),
                                  smoothing_function=SmoothingFunction().method1)

        return {"rouge_l": rouge_l, "bleu_4": bleu_score}
    except Exception as e:
        logging.warning(f"Error computing ROUGE/BLEU: {e}")
        return {"rouge_l": 0.0, "bleu_4": 0.0}

def evaluate_node_relevance_opensource(query: str, hierarchical_info: Dict[int, List[Dict]], sentence_model) -> float:
    """Evaluate relevance of retrieved graph nodes to the query using open source model."""
    try:
        relevant_nodes = 0
        total_nodes = 0
        
        # Encode query
        query_embedding = sentence_model.encode([query])
        
        for layer, nodes in hierarchical_info.items():
            for node in nodes:
                total_nodes += 1
                content = node.get('content', {}).get('summary', node.get('content', {}).get('description', ''))
                if not content:
                    continue
                
                # Encode node content
                content_embedding = sentence_model.encode([content])
                
                # Calculate similarity
                similarity = cosine_similarity(query_embedding, content_embedding)[0][0]
                
                # Threshold for relevance
                if similarity > 0.3:
                    relevant_nodes += 1

        return relevant_nodes / total_nodes if total_nodes > 0 else 0.0
    except Exception as e:
        logging.warning(f"Error evaluating node relevance: {e}")
        return 0.0

def evaluate_query(query_engine, query_data: Dict, evaluators, sentence_model) -> Dict[str, Any]:
    """Evaluate a single query against the ArchRAG query engine."""
    query = query_data["question"]
    reference = query_data["answer"]
    context = query_data.get("context", "")

    try:
        # Get response from query engine
        result = query_engine.query(query)

        # Handle different types of responses for text extraction
        if isinstance(result, dict):
            response_text = result.get("response", "")
            hierarchical_info = result.get("hierarchical_info", {})
        else:
            response_text = str(result)
            hierarchical_info = {}

        # Handle empty or invalid response
        if not response_text or response_text.startswith("I couldn't find relevant information"):
            logging.warning(f"Empty or invalid response for query: {query}")
            response_text = ""

        # Unpack evaluators
        faithfulness_evaluator, relevancy_evaluator, correctness_evaluator = evaluators

        # Prepare contexts for evaluation
        eval_contexts = []
        if hierarchical_info:
            for layer, nodes in hierarchical_info.items():
                for node in nodes:
                    content = node.get('content', {})
                    if isinstance(content, dict):
                        content = content.get('summary', content.get('description', ''))
                    else:
                        content = str(content)
                    if content:
                        eval_contexts.append(content)
        if not eval_contexts:
            eval_contexts = [context] if context else ["No context available"]

        # Evaluate faithfulness
        faithfulness_result = faithfulness_evaluator.evaluate(
            query=query,
            response=response_text,
            contexts=eval_contexts
        )

        # Evaluate relevancy
        relevancy_result = relevancy_evaluator.evaluate(
            query=query,
            response=response_text,
            contexts=eval_contexts
        )

        # Evaluate source node relevancy
        source_relevancy_results = []
        if hierarchical_info:
            for layer, nodes in hierarchical_info.items():
                for node in nodes:
                    content = node.get('content', {})
                    if isinstance(content, dict):
                        content = content.get('summary', content.get('description', ''))
                    else:
                        content = str(content)
                    if not content:
                        continue
                    try:
                        node_result = relevancy_evaluator.evaluate(
                            query=query,
                            response=response_text,
                            contexts=[content]
                        )
                        source_relevancy_results.append(node_result.passing)
                    except Exception as e:
                        logging.warning(f"Error evaluating node relevancy: {e}")
                        continue

        source_relevancy_score = (sum(source_relevancy_results) / len(source_relevancy_results)
                                 if source_relevancy_results else 0.0)

        # Evaluate correctness
        correctness_result = correctness_evaluator.evaluate(
            query=query,
            response=response_text,
            reference=reference
        )

        # Compute ROUGE and BLEU
        rouge_bleu_scores = compute_rouge_bleu(response_text, reference)

        # Evaluate node relevance
        node_precision = evaluate_node_relevance_opensource(query, hierarchical_info, sentence_model) if hierarchical_info else 0.0

        # Compute community coverage
        community_coverage = 0.0
        if hierarchical_info:
            valid_layers = [l for l, nodes in hierarchical_info.items() if nodes]
            community_coverage = len(valid_layers) / (len(hierarchical_info) or 1)

        return {
            "query": query,
            "response": response_text,
            "reference": reference,
            "faithfulness_score": 1 if faithfulness_result.passing else 0,
            "faithfulness_feedback": faithfulness_result.feedback or "No feedback available",
            "relevancy_score": 1 if relevancy_result.passing else 0,
            "relevancy_feedback": relevancy_result.feedback or "No feedback available",
            "source_relevancy_score": source_relevancy_score,
            "correctness_score": correctness_result.score if hasattr(correctness_result, 'score') else 0.0,
            "correctness_passing": 1 if correctness_result.passing else 0,
            "correctness_feedback": correctness_result.feedback or "No feedback available",
            "rouge_l": rouge_bleu_scores["rouge_l"],
            "bleu_4": rouge_bleu_scores["bleu_4"],
            "node_precision": node_precision,
            "community_coverage": community_coverage
        }

    except Exception as e:
        logging.error(f"Error evaluating query '{query}': {e}")
        import traceback
        logging.error(f"Traceback: {traceback.format_exc()}")
        return {
            "query": query,
            "response": "",
            "reference": reference,
            "faithfulness_score": 0,
            "faithfulness_feedback": f"Evaluation error: {e}",
            "relevancy_score": 0,
            "relevancy_feedback": f"Evaluation error: {e}",
            "source_relevancy_score": 0.0,
            "correctness_score": 0.0,
            "correctness_passing": 0,
            "correctness_feedback": f"Evaluation error: {e}",
            "rouge_l": 0.0,
            "bleu_4": 0.0,
            "node_precision": 0.0,
            "community_coverage": 0.0
        }

def create_visualizations(results_df: pd.DataFrame, metrics: Dict[str, float]):
    """Create visualizations for evaluation metrics."""
    try:
        valid_metrics = {k: v for k, v in metrics.items() if not pd.isna(v)}
        metric_names = list(valid_metrics.keys())
        metric_values = list(valid_metrics.values())

        plt.figure(figsize=(12, 6))
        bars = plt.bar(metric_names, metric_values)
        plt.ylim(0, 1.1)
        plt.title('ArchRAG Evaluation Metrics (Open Source)')
        plt.ylabel('Score')
        plt.xticks(rotation=45)
        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                     f'{height:.2f}', ha='center', va='bottom')
        plt.tight_layout()
        plt.savefig('archrag_evaluation_metrics.png')
        plt.close()

        if 'correctness_score' in results_df.columns:
            plt.figure(figsize=(8, 6))
            plt.hist(results_df['correctness_score'].dropna(), bins=10, alpha=0.7)
            plt.title('Distribution of Correctness Scores')
            plt.xlabel('Correctness Score')
            plt.ylabel('Count')
            plt.savefig('correctness_distribution.png')
            plt.close()

        if 'node_precision' in results_df.columns:
            plt.figure(figsize=(8, 6))
            plt.hist(results_df['node_precision'].dropna(), bins=10, alpha=0.7)
            plt.title('Distribution of Node Retrieval Precision')
            plt.xlabel('Node Precision')
            plt.ylabel('Count')
            plt.savefig('node_precision_distribution.png')
            plt.close()

        if 'rouge_l' in results_df.columns and 'node_precision' in results_df.columns:
            plt.figure(figsize=(8, 6))
            plt.scatter(results_df['rouge_l'], results_df['node_precision'], alpha=0.5)
            plt.title('ROUGE-L vs Node Retrieval Precision')
            plt.xlabel('ROUGE-L Score')
            plt.ylabel('Node Precision')
            plt.savefig('rouge_vs_node.png')
            plt.close()
    except Exception as e:
        logging.error(f"Error creating visualizations: {e}")

def evaluate_archrag_system(pipeline, dataset_path: str, config: Dict[str, str], max_docs: int = 10):
    """Main function to evaluate the ArchRAG system with open source models."""
    logging.info("Starting ArchRAG service evaluation with open source models...")

    # Step 1: Log PDF content
    logging.info("Checking PDF content...")
    pdf_texts = log_pdf_content(config["pdf_dir"])

    # Step 2: Sample dataset
    logging.info("Sampling dataset...")
    dataset_samples = sample_dataset(dataset_path)

    # Step 3: Build knowledge graph
    logging.info(f"Building knowledge graph with max_docs={max_docs}...")
    pipeline.build_knowledge_graph(max_docs=max_docs)
    query_engine = pipeline.query_engine

    # Step 4: Check Neo4j graph
    logging.info("Checking Neo4j graph...")
    graph_stats = check_neo4j_graph(config["neo4j_uri"], config["neo4j_username"], config["neo4j_password"])

    # Step 5: Test query engine
    logging.info("Testing query engine...")
    test_query_engine(query_engine)

    # Load evaluation dataset
    eval_dataset = load_eval_dataset(json_file_path=dataset_path)

    # Initialize evaluators with open source models
    evaluators = init_evaluators()
    sentence_model = SentenceTransformer('all-MiniLM-L6-v2')

    # Run evaluations
    results = []
    for query_data in tqdm(eval_dataset, desc="Evaluating queries"):
        result = evaluate_query(query_engine, query_data, evaluators, sentence_model)
        results.append(result)

    # Convert to DataFrame
    results_df = pd.DataFrame(results)

    # Calculate aggregate metrics
    metrics = {
        "Faithfulness": results_df["faithfulness_score"].mean(),
        "Relevancy": results_df["relevancy_score"].mean(),
        "Source Relevancy": results_df["source_relevancy_score"].mean()
                          if not results_df["source_relevancy_score"].empty else 0.0,
        "Correctness Score": results_df["correctness_score"].mean(),
        "Correctness Pass Rate": results_df["correctness_passing"].mean(),
        "ROUGE-L": results_df["rouge_l"].mean(),
        "BLEU-4": results_df["bleu_4"].mean(),
        "Node Precision": results_df["node_precision"].mean(),
        "Community Coverage": results_df["community_coverage"].mean()
    }

    # Create metrics summary DataFrame
    metrics_df = pd.DataFrame({
        "Metric": list(metrics.keys()),
        "Score": list(metrics.values())
    })

    # Generate visualizations
    create_visualizations(results_df, metrics)

    # Save results
    results_df.to_csv("archrag_evaluation_detailed_results.csv", index=False)
    metrics_df.to_csv("archrag_evaluation_metrics.csv", index=False)

    logging.info("\nArchRAG Evaluation Summary (Open Source):")
    logging.info(metrics_df.to_string(index=False))
    logging.info("\nDetailed results saved to 'archrag_evaluation_detailed_results.csv'")
    logging.info("Metrics summary saved to 'archrag_evaluation_metrics.csv'")
    logging.info("Visualizations saved as PNG files")

    return results_df, metrics_df, graph_stats, pdf_texts


# Usage example (compatible with your existing code)
if __name__ == "__main__":
    config = {
        "pdf_dir": "/home/871323/RAG/test/150papers/150papers",
        "neo4j_uri": os.getenv("NEO4J_URI"),
        "neo4j_username": os.getenv("NEO4J_USERNAME", "neo4j"),
        "neo4j_password": os.getenv("NEO4J_PASSWORD"),
        # Note: openai_api_key is no longer needed for open source evaluation
    }
    
    # Note: ArchRAGPipeline must be imported or defined
    # from your_pipeline_file import ArchRAGPipeline
    rpipeline = ArchRAGPipeline(**config)
    
    results_df, metrics_df, graph_stats, pdf_texts = evaluate_archrag_system(
        pipeline=rpipeline,
        dataset_path="/home/871323/RAG/test/100diabetes_qa_dataset.json",
        config=config,
        max_docs=150
    )

INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cuda:0
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: all-MiniLM-L6-v2


Error loading quantized model: Failed to import transformers.integrations.bitsandbytes because of the following error (look up to see its traceback):
cannot import name 'tarfile' from 'backports' (/home/ViperAppsFiles/python/anaconda/miniconda/3.10/lib/python3.10/site-packages/backports/__init__.py)
Falling back to non-quantized model...


INFO:accelerate.utils.modeling:We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

INFO:root:Starting ArchRAG service evaluation with open source models...
INFO:root:Checking PDF content...
INFO:root:Extracting text from PDF: 23704672.pdf
INFO:root:Sample text from 23704672.pdf: Rapid Rise in Hypertension and
Nephropathy in Youth With
Type 2 Diabetes
The TODAY clinical trial
TODAY S TUDY GROUP*
OBJECTIVE dAmong adolescents with type 2 diabetes, there is limited information regarding
incidence and progression of hypertension and microalbuminuria. Hypertension and micro-
albuminuria assessments made during the TODAY clinical trial were analyzed for effect of treat-
ment, glycemic control, sex, and race/ethnicity.
RESEARCH DESIGN AND METHODS dA cohort of 699 adolescents, 1...
INFO:root:Extracting text from PDF: 24170750.pdf
INFO:root:Sample text from 24170750.pdf: Reductions in Regimen Distress
Are Associated With ImprovedManagement and Glycemic
Control Over Time
OBJECTIVE
Cross-sectional and longitudinal associations among regimen distress (RD), self-
management, and g

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating queries: 100%|██████████| 100/100 [09:00<00:00,  5.40s/it]
INFO:root:
ArchRAG Evaluation Summary (Open Source):
INFO:root:               Metric    Score
         Faithfulness 0.640000
            Relevancy 0.940000
     Source Relevancy 0.940000
    Correctness Score 0.341834
Correctness Pass Rate 0.750000
              ROUGE-L 0.095578
               BLEU-4 0.008635
       Node Precision 0.082727
   Community Coverage 1.000000
INFO:root:
Detailed results saved to 'archrag_evaluation_detailed_results.csv'
INFO:root:Metrics summary saved to 'archrag_evaluation_metrics.csv'
INFO:root:Visualizations saved as PNG files
